# CS781 Capstone - Phase 1, Submission 1
## Reproducible legal-language-model experiment

This notebook is an end-to-end Phase 1 implementation. It validates and splits the official corpus, trains and fingerprints a byte-level BPE tokenizer, constructs an independently defined decoder-only Transformer, trains with reproducible mixed-source sampling and resumable checkpoints, evaluates on a fixed local development split, generates the official 128-word continuations, and packages both the Codabench submission and a recovery archive.

The submitted intervention is a SwiGLU feed-forward sublayer in an otherwise conventional Pre-LayerNorm decoder-only Transformer. The recorded control is a parameter-matched GELU feed-forward model with the same tokenizer, data split, source mixture, context length, training-token budget, optimizer, schedule, and evaluator.

Long-running actions remain explicitly gated: attaching the official files enables schema-verified data loading, while full training and official generation require their dedicated Boolean switches.


## Execution map

Fresh session: run sections 1-9 to discover and validate data, train or restore the tokenizer, build the model, and run smoke tests. Set `CFG.full_training_authorized = True` only after reviewing every manifest, then enable `RUN_FULL_TRAINING` in Cell 29. After selecting a checkpoint, set the evaluator path if discovery is ambiguous and enable `RUN_INFERENCE` in Cell 35. Packaging, recovery export, and final verification then use the generated artifacts.

Crash resume: rerun setup, data, tokenizer, and model cells with the same configuration. The training function verifies the tokenizer, dataset, architecture, training configuration, history hash, sampler cursor, optimizer, scheduler, scaler, and RNG state before resuming. `num_workers=0` is enforced so a saved sampler cursor represents consumed batches rather than prefetched work.


## 1. Environment setup and reproducibility

In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import sys
import tempfile
import time
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Mapping, Sequence

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, Sampler

try:
    from tqdm.auto import tqdm
except ImportError as exc:
    raise RuntimeError("tqdm is required; install it in the Kaggle environment") from exc

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def atomic_write_bytes(path: Path, payload: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
        temporary = Path(handle.name)
    os.replace(temporary, path)

def atomic_write_json(path: Path, value: Any) -> None:
    atomic_write_bytes(Path(path), (json.dumps(value, indent=2, ensure_ascii=False) + "\n").encode("utf-8"))

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

print({"python": sys.version.split()[0], "torch": torch.__version__, "utc": utc_now()})


In [ ]:
# Experiment configuration. Training-field names are resolved only from attached JSON schemas.
@dataclass
class RunConfig:
    experiment_id: str = "phase1_submission01_swiglu"
    seed: int = 781
    workspace: str = field(default_factory=lambda: str(Path('/kaggle/working/CS781_Capstone/phase1/submission_01') if Path('/kaggle/working').exists() else Path.cwd()/'CS781_Capstone/phase1/submission_01'))
    input_root: str = field(default_factory=lambda: str(Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()))
    train_files: tuple[str,...] = ('train_acts.json','train_high_courts.json','train_supreme_court.json','train_tribunals.json')
    test_file: str = 'Phase1- Test.json'
    test_filename_aliases: tuple[str,...] = ('Phase1- Test.json','test.json')
    hf_repo_id: str = 'Exploration-Lab/CS-781-Capstone'
    hf_allow_patterns: tuple[str,...] = ('*Phase_1_Pretraining/*','*Phase-1_Pretraining/*')
    official_file_overrides: dict[str,str] = field(default_factory=dict)
    # Every schema field remains unset until Cell 10 verifies the attached official files.
    document_id_field: str | None = None
    document_text_field: str | None = None
    test_id_field: str | None = None
    test_prompt_field: str | None = None
    validation_fraction: float | None = 0.05
    source_mixing_ratios: dict[str,float] | None = field(default_factory=lambda: {
        'acts': 0.25,
        'high_courts': 0.25,
        'supreme_court': 0.25,
        'tribunals': 0.25,
    })
    architecture_record: dict[str,Any] = field(default_factory=lambda: {
        'family': 'decoder-only causal Transformer',
        'intervention': 'SwiGLU feed-forward sublayer',
        'normalization': 'Pre-LayerNorm',
        'position_encoding': 'learned absolute embeddings',
        'embedding_tying': True,
        'blocks': 8,
        'hidden_size': 768,
        'heads': 12,
        'ffn_hidden_size': 2048,
        'context_length': 1024,
        'parameter_range': [50_000_000, 300_000_000],
    })
    innovation_hypothesis: str = (
        'At a matched training-token budget, replacing the conventional GELU feed-forward sublayer '
        'with a parameter-matched SwiGLU sublayer will improve fixed-development next-token loss and '
        'official continuation metrics for heterogeneous Indian legal text.'
    )
    ablation_control: str = (
        'A parameter-matched Pre-LayerNorm decoder using a conventional GELU feed-forward sublayer; '
        'tokenizer, split, source mixture, context, optimizer, schedule, token budget and evaluator remain fixed.'
    )
    full_training_authorized: bool = False
    trained_from_scratch_attestation: bool = False
    decoding: dict[str,Any] = field(default_factory=lambda: {
        'strategy': 'greedy', 'target_words': 128, 'max_total_new_tokens': 2048
    })

CFG = RunConfig()
ROOT = Path(CFG.workspace)
DIRS = {name:ROOT/name for name in ('model','tokenizer','data','checkpoints','histories','evaluations','predictions','plots','recovery','manifests','submission')}
for folder in DIRS.values():
    folder.mkdir(parents=True,exist_ok=True)

RUN_CONFIG_PATH = DIRS['manifests']/'run_config.json'
def register_run_config(cfg: RunConfig, *, allow_initial: bool=True):
    # Normalize tuples and other JSON-compatible containers before comparing a reloaded manifest.
    current = json.loads(json.dumps(asdict(cfg),ensure_ascii=False))
    stable = (
        'experiment_id','seed','train_files','test_file','test_filename_aliases','hf_repo_id',
        'hf_allow_patterns','official_file_overrides','document_id_field','document_text_field',
        'test_id_field','test_prompt_field','validation_fraction','source_mixing_ratios',
        'architecture_record','innovation_hypothesis','ablation_control'
    )
    if RUN_CONFIG_PATH.exists():
        existing = json.loads(RUN_CONFIG_PATH.read_text(encoding='utf-8'))
        merged = dict(existing)
        conflicts = {}
        for key in stable:
            old, new = existing.get(key), current.get(key)
            # A null field in an initial discovery manifest may be filled exactly once after schema verification.
            if old is None and new is not None:
                merged[key] = new
            elif new is not None and old != new:
                conflicts[key] = (old, new)
        if conflicts:
            raise RuntimeError(f'Existing run config conflict: {conflicts}. Use the matching configuration or a new workspace.')
        if merged != existing:
            atomic_write_json(RUN_CONFIG_PATH, merged)
        return merged
    if allow_initial:
        atomic_write_json(RUN_CONFIG_PATH, current)
    return current

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)
print('Workspace:', ROOT)
print('Training schema fields will be committed after Cell 10 verifies the official JSON files.')


## 2. Kaggle GPU detection and optional authentication

In [ ]:
def gpu_inventory() -> list[dict[str, Any]]:
    rows = []
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        rows.append({
            "index": index,
            "device": f"cuda:{index}",
            "name": props.name,
            "memory_gib": round(props.total_memory / 2**30, 2),
            "capability": f"{props.major}.{props.minor}",
        })
    return rows

GPUS = gpu_inventory()
print(json.dumps(GPUS, indent=2))
if not GPUS:
    print("WARNING: no CUDA GPU detected. Do not start full training.")
elif len(GPUS) == 1:
    print("One independent experiment can use cuda:0.")
else:
    print("Independent jobs may be assigned cuda:0 and cuda:1; this notebook does not use DataParallel.")

In [ ]:
# Optional only when an attached artifact actually requires Hugging Face authentication.
# Do not paste a token into the notebook. Use a Kaggle secret named HF_TOKEN.
def optional_hf_login(secret_name: str = "HF_TOKEN") -> bool:
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import login
        token = UserSecretsClient().get_secret(secret_name)
        if not token:
            return False
        login(token=token, add_to_git_credential=False)
        return True
    except Exception as exc:
        print(f"HF login skipped: {type(exc).__name__}: {exc}")
        return False

# optional_hf_login()  # Uncomment only if required.

## 3. Official file discovery and schema inspection

In [ ]:
import subprocess

HF_DATA_ROOT=Path('/kaggle/working/capstone_data') if Path('/kaggle/working').exists() else ROOT/'downloaded_official_data'
INSTALL_RUNTIME_DEPENDENCIES=False
DOWNLOAD_PHASE1_FROM_HF=False

def ensure_runtime_dependency(import_name:str,pip_spec:str)->None:
    if importlib.util.find_spec(import_name) is not None:
        return
    if not INSTALL_RUNTIME_DEPENDENCIES:
        raise RuntimeError(
            f'Missing {import_name!r}. Set INSTALL_RUNTIME_DEPENDENCIES=True and rerun this cell '
            f'to install {pip_spec!r} into the Kaggle session.'
        )
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pip_spec])

def kaggle_hf_token(secret_name:str='HF_TOKEN')->str|None:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name) or None
    except Exception:
        return None

def inspect_huggingface_phase1_files(repo_id:str)->list[str]:
    ensure_runtime_dependency('huggingface_hub','huggingface_hub>=0.25,<2')
    from huggingface_hub import HfApi
    api=HfApi(token=kaggle_hf_token())
    entries=list(api.list_repo_tree(repo_id=repo_id,repo_type='dataset',recursive=True))
    phase1=[];total_bytes=0
    for entry in entries:
        name=str(getattr(entry,'path',''))
        if 'phase_1' in name.lower() or 'phase-1' in name.lower():
            size=int(getattr(entry,'size',0) or 0);phase1.append(name);total_bytes+=size
            print(f'{name} ({size/2**30:.2f} GiB)' if size else name)
    if not phase1:
        raise RuntimeError(f'No Phase 1 files were visible in {repo_id}; check access/authentication')
    free=shutil.disk_usage(HF_DATA_ROOT.parent if HF_DATA_ROOT.parent.exists() else Path.cwd()).free
    print({'phase1_download_gib':round(total_bytes/2**30,2),'destination_free_gib':round(free/2**30,2)})
    if total_bytes and free<total_bytes+8*2**30:
        print('WARNING: direct download leaves less than 8 GiB for compressed preprocessing and checkpoints; prefer a private attached Kaggle dataset.')
    return phase1

def download_huggingface_phase1(cfg:RunConfig,destination:Path)->Path:
    ensure_runtime_dependency('huggingface_hub','huggingface_hub>=0.25,<2')
    ensure_runtime_dependency('ijson','ijson>=3.3,<4')
    from huggingface_hub import snapshot_download
    published=inspect_huggingface_phase1_files(cfg.hf_repo_id)
    required=set(cfg.train_files)
    available_names={Path(name).name for name in published}
    missing=sorted(required-available_names)
    if missing: raise RuntimeError(f'Official repository listing is missing required training files: {missing}')
    if not (set(cfg.test_filename_aliases)&available_names): raise RuntimeError(f'No official test filename matched {cfg.test_filename_aliases}')
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    resolved=snapshot_download(
        repo_id=cfg.hf_repo_id,repo_type='dataset',token=kaggle_hf_token(),
        allow_patterns=list(cfg.hf_allow_patterns),local_dir=str(destination),
    )
    print('Downloaded Phase 1 snapshot to',resolved)
    return Path(resolved)

print({'data_search_roots':[str(Path(CFG.input_root)),str(HF_DATA_ROOT)],'hf_destination_free_gib':round(shutil.disk_usage(HF_DATA_ROOT.parent if HF_DATA_ROOT.parent.exists() else Path.cwd()).free/2**30,2)})
if DOWNLOAD_PHASE1_FROM_HF:
    download_huggingface_phase1(CFG,HF_DATA_ROOT)

def discover_named_files(roots:Sequence[Path],cfg:RunConfig)->dict[str,list[Path]]:
    wanted=set(cfg.train_files)
    aliases=set(cfg.test_filename_aliases)
    found={name:[] for name in [*cfg.train_files,cfg.test_file]}
    for root in roots:
        root=Path(root)
        if not root.exists(): continue
        for path in root.rglob('*'):
            if not path.is_file(): continue
            if path.name in wanted:
                found[path.name].append(path.resolve())
            elif path.name in aliases:
                found[cfg.test_file].append(path.resolve())
    for name in found:
        found[name]=sorted(set(found[name]))
    return found

DATA_SEARCH_ROOTS=[]
for candidate in (Path(CFG.input_root),HF_DATA_ROOT):
    resolved=candidate.resolve() if candidate.exists() else candidate
    if resolved not in DATA_SEARCH_ROOTS:
        DATA_SEARCH_ROOTS.append(resolved)
DISCOVERED=discover_named_files(DATA_SEARCH_ROOTS,CFG)
for name,paths in sorted(DISCOVERED.items()):
    print(name,'->',[str(path) for path in paths])

def require_unique_file(discovered:Mapping[str,Sequence[Path]],name:str)->Path:
    override=CFG.official_file_overrides.get(name)
    if override:
        selected=Path(override).expanduser().resolve()
        if not selected.is_file():
            raise FileNotFoundError(f'Configured override for {name} does not exist: {selected}')
        return selected
    matches=list(discovered.get(name,[]))
    if len(matches)!=1:
        raise RuntimeError(
            f'Expected exactly one {name!r}; found {len(matches)}: {matches}. '
            'Set CFG.official_file_overrides[name] to an inspected exact path; duplicates are never selected silently.'
        )
    return Path(matches[0])


In [ ]:
MAX_IN_MEMORY_JSON_BYTES=256*2**20

def _ijson():
    ensure_runtime_dependency('ijson','ijson>=3.3,<4')
    import ijson
    return ijson

def detect_json_layout(path:Path)->tuple[str,str|None]:
    ijson=_ijson()
    with Path(path).open('rb') as handle:
        parser=ijson.parse(handle)
        try: prefix,event,value=next(parser)
        except StopIteration: raise ValueError(f'Empty JSON file: {path}')
        if event=='start_array': return 'array',None
        if event!='start_map': raise TypeError(f'Unsupported JSON root event {event}: {path}')
        pending_key=None
        for prefix,event,value in parser:
            if prefix=='' and event=='map_key':
                pending_key=str(value);continue
            if pending_key is not None:
                if event=='start_array': return 'wrapper_array',pending_key
                if event in ('start_map','string','number','boolean','null'):
                    return 'id_keyed_object',None
        return 'empty_object',None

def iter_json_records(path:Path,*,id_field:str|None=None,text_field:str|None=None):
    ijson=_ijson();layout,wrapper=detect_json_layout(path)
    with Path(path).open('rb') as handle:
        if layout=='array':
            iterator=ijson.items(handle,'item')
            for row in iterator:
                if not isinstance(row,dict): raise TypeError('Array records must be JSON objects')
                yield row
        elif layout=='wrapper_array':
            for row in ijson.items(handle,f'{wrapper}.item'):
                if not isinstance(row,dict): raise TypeError('Wrapped records must be JSON objects')
                yield row
        elif layout=='id_keyed_object':
            for key,value in ijson.kvitems(handle,''):
                if isinstance(value,dict):
                    row=dict(value)
                    if id_field and id_field not in row: row[id_field]=str(key)
                    row.setdefault('_container_key',str(key))
                    yield row
                elif isinstance(value,str):
                    yield {
                        id_field or '_container_key':str(key),
                        text_field or '_container_value':value,
                        '_container_key':str(key),
                    }
                else:
                    raise TypeError(f'Unsupported value under top-level ID {key!r}: {type(value).__name__}')
        elif layout!='empty_object':
            raise TypeError(f'Unsupported JSON layout: {layout}')

def load_json_container(path:Path)->Any:
    path=Path(path)
    if path.stat().st_size>MAX_IN_MEMORY_JSON_BYTES:
        raise MemoryError(f'{path} is too large for json.load(); use iter_json_records()')
    with path.open('r',encoding='utf-8-sig') as handle: return json.load(handle)

def records_from_container(container:Any,*,id_field:str|None=None,text_field:str|None=None)->list[dict[str,Any]]:
    if isinstance(container,list): records=container
    elif isinstance(container,dict):
        if container and all(isinstance(value,dict) for value in container.values()):
            records=[]
            for key,value in container.items():
                row=dict(value)
                if id_field and id_field not in row: row[id_field]=str(key)
                row.setdefault('_container_key',str(key));records.append(row)
        elif container and all(isinstance(value,str) for value in container.values()):
            id_name,text_name=id_field or '_container_key',text_field or '_container_value'
            records=[{id_name:str(key),text_name:value,'_container_key':str(key)} for key,value in container.items()]
        else:
            candidates=[(key,value) for key,value in container.items() if isinstance(value,list)]
            if len(candidates)!=1: raise ValueError(f'Ambiguous JSON object; list keys: {[key for key,_ in candidates]}')
            records=candidates[0][1]
    else: raise TypeError(f'Unsupported root: {type(container).__name__}')
    if not all(isinstance(record,dict) for record in records): raise TypeError('All records must be JSON objects')
    return records

def sample_json_records(path:Path,limit:int=256)->list[dict[str,Any]]:
    import itertools
    return list(itertools.islice(iter_json_records(path),int(limit)))

def inspect_schema(path:Path,sample_size:int=3)->dict[str,Any]:
    rows=sample_json_records(path,max(256,sample_size))
    keys=Counter(key for row in rows for key in row)
    samples=[{key:{'type':type(value).__name__,'preview':str(value)[:140]} for key,value in row.items()} for row in rows[:sample_size]]
    layout,wrapper=detect_json_layout(path)
    return {
        'path':str(path),'bytes':Path(path).stat().st_size,'layout':layout,'wrapper_key':wrapper,
        'sampled_records':len(rows),'sample_field_presence':dict(keys),'samples':samples,
    }

def _verified_field(rows:Sequence[Mapping[str,Any]],candidates:Sequence[str],*,unique:bool=False)->str:
    if '_container_key' in candidates:
        values=[row.get('_container_key') for row in rows]
        if values and all(value is not None and str(value).strip() for value in values):
            if not unique or len({str(value) for value in values})==len(values): return '_container_key'
    valid=[]
    for name in candidates:
        if name=='_container_key': continue
        values=[row.get(name) for row in rows]
        if values and all(value is not None and str(value).strip() for value in values):
            if not unique or len({str(value) for value in values})==len(values): valid.append(name)
    if len(valid)!=1: raise RuntimeError(f'Expected exactly one verified field from {list(candidates)}, found {valid}')
    return valid[0]

def infer_phase1_record_fields(path:Path,*,role:str)->tuple[str,str]:
    rows=sample_json_records(path,256)
    if not rows: raise ValueError(f'No records in {path}')
    id_field=_verified_field(rows,('_container_key','document_id','doc_id','case_id','uid','id'),unique=True)
    text_candidates=('_container_value','text','document','content','case_text','judgment','judgement','prompt')
    common=[name for name in text_candidates if all(isinstance(row.get(name),str) and row.get(name).strip() for row in rows)]
    if len(common)!=1:
        raise RuntimeError(f'{role} text field is ambiguous in {path}: {common}. Inspect SCHEMAS and set an explicit parser.')
    return id_field,common[0]

def resolve_official_schema_fields(cfg:RunConfig,discovered:Mapping[str,Sequence[Path]])->dict[str,str]:
    train_pairs={infer_phase1_record_fields(require_unique_file(discovered,name),role='training') for name in cfg.train_files}
    if len(train_pairs)!=1: raise RuntimeError(f'Training files do not share one schema: {sorted(train_pairs)}')
    document_id_field,document_text_field=next(iter(train_pairs))
    test_id_field,test_prompt_field=infer_phase1_record_fields(require_unique_file(discovered,cfg.test_file),role='test')
    return dict(document_id_field=document_id_field,document_text_field=document_text_field,test_id_field=test_id_field,test_prompt_field=test_prompt_field)

_schema_names=[*CFG.train_files,CFG.test_file]
if all(len(DISCOVERED.get(name,[]))==1 or name in CFG.official_file_overrides for name in _schema_names):
    SCHEMAS={name:inspect_schema(require_unique_file(DISCOVERED,name)) for name in _schema_names}
    print(json.dumps(SCHEMAS,indent=2,ensure_ascii=False))
    RESOLVED_SCHEMA=resolve_official_schema_fields(CFG,DISCOVERED)
    for key,value in RESOLVED_SCHEMA.items():
        configured=getattr(CFG,key)
        if configured is not None and configured!=value: raise RuntimeError(f'Configured {key}={configured!r}, official sample verifies {value!r}')
        setattr(CFG,key,value)
    print('Verified sampled field mapping; every record is revalidated during preprocessing:',RESOLVED_SCHEMA)
else:
    SCHEMAS={};RESOLVED_SCHEMA=None
    missing={name:len(DISCOVERED.get(name,[])) for name in _schema_names if len(DISCOVERED.get(name,[]))!=1 and name not in CFG.official_file_overrides}
    print('Schema inspection deferred until official files/overrides resolve uniquely:',missing)


## 4. Validated loading, provenance, deduplication, and document-level split

In [ ]:
import zlib

@dataclass(frozen=True)
class Document:
    uid:str
    text:str|None
    source:str
    raw_id:str
    content_hash:str
    word_count:int
    store_path:str|None=None
    store_offset:int|None=None
    store_length:int|None=None

def normalize_text(value:Any)->str:
    if not isinstance(value,str): raise TypeError(f'Text must be a string, got {type(value).__name__}')
    result=value.replace('\r\n','\n').strip()
    if not result: raise ValueError('Empty text')
    return result

def document_text(document:Document)->str:
    if document.text is not None: return document.text
    if document.store_path is None or document.store_offset is None or document.store_length is None:
        raise RuntimeError(f'Document has no text storage: {document.uid}')
    with Path(document.store_path).open('rb') as handle:
        handle.seek(document.store_offset);payload=handle.read(document.store_length)
    return zlib.decompress(payload).decode('utf-8')

def _load_document_index(index_path:Path,store_path:Path)->list[Document]:
    documents=[]
    with Path(index_path).open('r',encoding='utf-8') as handle:
        for line in handle:
            row=json.loads(line)
            documents.append(Document(
                uid=row['uid'],text=None,source=row['source'],raw_id=row['raw_id'],
                content_hash=row['content_hash'],word_count=int(row['word_count']),
                store_path=str(store_path),store_offset=int(row['store_offset']),store_length=int(row['store_length']),
            ))
    return documents

def load_documents(path:Path,source:str,id_field:str,text_field:str,store_dir:Path|None=None)->tuple[list[Document],list[dict[str,Any]]]:
    path=Path(path);source_sha=sha256_file(path)
    if store_dir is None:
        accepted=[];rejected=[]
        for index,row in enumerate(iter_json_records(path,id_field=id_field,text_field=text_field)):
            try:
                if id_field not in row or text_field not in row: raise KeyError(f'Missing required fields {id_field}/{text_field}')
                raw_id=str(row[id_field]).strip()
                if not raw_id: raise ValueError('Empty identifier')
                text=normalize_text(row[text_field]);digest=hashlib.sha256(text.encode()).hexdigest()
                accepted.append(Document(f'{source}:{raw_id}',text,source,raw_id,digest,len(text.split())))
            except Exception as exc: rejected.append({'index':index,'error':f'{type(exc).__name__}: {exc}'})
        return accepted,rejected

    store_dir=Path(store_dir);store_dir.mkdir(parents=True,exist_ok=True)
    stem=f'{source}.{source_sha[:16]}'
    store_path=store_dir/f'{stem}.zbin';index_path=store_dir/f'{stem}.index.jsonl';meta_path=store_dir/f'{stem}.meta.json'
    expected_meta={'source_file':str(path.resolve()),'source_sha256':source_sha,'id_field':id_field,'text_field':text_field,'format_version':1}
    if store_path.exists() and index_path.exists() and meta_path.exists():
        meta=json.loads(meta_path.read_text(encoding='utf-8'))
        if all(meta.get(key)==value for key,value in expected_meta.items()):
            return _load_document_index(index_path,store_path),list(meta.get('rejections',[]))

    with tempfile.NamedTemporaryFile(dir=store_dir,delete=False) as handle: temporary_store=Path(handle.name)
    with tempfile.NamedTemporaryFile(dir=store_dir,delete=False) as handle: temporary_index=Path(handle.name)
    accepted_count=0;rejected=[]
    try:
        with temporary_store.open('wb') as store_handle, temporary_index.open('w',encoding='utf-8') as index_handle:
            for index,row in enumerate(iter_json_records(path,id_field=id_field,text_field=text_field)):
                if index and index%10_000==0: print(f'{source}: preprocessed {index:,} records')
                try:
                    if id_field not in row or text_field not in row: raise KeyError(f'Missing required fields {id_field}/{text_field}')
                    raw_id=str(row[id_field]).strip()
                    if not raw_id: raise ValueError('Empty identifier')
                    text=normalize_text(row[text_field]);encoded=text.encode('utf-8')
                    compressed=zlib.compress(encoded,level=3);offset=store_handle.tell();store_handle.write(compressed)
                    record={
                        'uid':f'{source}:{raw_id}','source':source,'raw_id':raw_id,
                        'content_hash':hashlib.sha256(encoded).hexdigest(),'word_count':len(text.split()),
                        'store_offset':offset,'store_length':len(compressed),
                    }
                    index_handle.write(json.dumps(record,ensure_ascii=False)+'\n');accepted_count+=1
                except Exception as exc: rejected.append({'index':index,'error':f'{type(exc).__name__}: {exc}'})
            store_handle.flush();os.fsync(store_handle.fileno());index_handle.flush();os.fsync(index_handle.fileno())
        # Replace only after both temporary handles are closed (required on Windows and safe on Kaggle/Linux).
        os.replace(temporary_store,store_path);os.replace(temporary_index,index_path)
    finally:
        temporary_store.unlink(missing_ok=True);temporary_index.unlink(missing_ok=True)
    meta={**expected_meta,'accepted':accepted_count,'rejected':len(rejected),'rejections':rejected,'created_utc':utc_now()}
    atomic_write_json(meta_path,meta)
    return _load_document_index(index_path,store_path),rejected

def corpus_audit(docs:Sequence[Document])->dict[str,Any]:
    ids=Counter(document.uid for document in docs);hashes=Counter(document.content_hash for document in docs)
    return {
        'documents':len(docs),'sources':dict(Counter(document.source for document in docs)),
        'whitespace_word_count':sum(document.word_count for document in docs),
        'words_by_source':dict(Counter({source:sum(document.word_count for document in docs if document.source==source) for source in {document.source for document in docs}})),
        'duplicate_uids':[key for key,value in ids.items() if value>1],
        'duplicate_content_hashes':[key for key,value in hashes.items() if value>1],
    }

def deduplicate_exact_documents(docs:Sequence[Document])->tuple[list[Document],list[dict[str,str]]]:
    kept=[];dropped=[];seen_ids={};seen_hashes={}
    for document in sorted(docs,key=lambda value:(value.uid,value.content_hash)):
        reason='duplicate_uid' if document.uid in seen_ids else 'duplicate_exact_text' if document.content_hash in seen_hashes else None
        if reason:
            retained=seen_ids.get(document.uid) if reason=='duplicate_uid' else seen_hashes[document.content_hash]
            dropped.append({'uid':document.uid,'kept_uid':retained,'reason':reason});continue
        kept.append(document);seen_ids[document.uid]=document.uid;seen_hashes[document.content_hash]=document.uid
    return kept,dropped

def dataset_fingerprint(docs:Sequence[Document])->str:
    digest=hashlib.sha256()
    for document in sorted(docs,key=lambda value:value.uid): digest.update(f'{document.uid}\t{document.content_hash}\n'.encode())
    return digest.hexdigest()


In [ ]:
def deterministic_document_split(documents:Sequence[Document],validation_fraction:float,seed:int)->tuple[list[Document],list[Document],dict[str,Any]]:
    if not 0<validation_fraction<1: raise ValueError('Choose validation_fraction strictly between zero and one')
    audit=corpus_audit(documents)
    if audit['duplicate_uids'] or audit['duplicate_content_hashes']: raise ValueError('Deduplicate exact documents before splitting')
    train=[];validation=[];per_source={}
    for source in sorted({document.source for document in documents}):
        group=sorted((document for document in documents if document.source==source),key=lambda value:value.uid)
        random.Random(seed+int(hashlib.sha256(source.encode()).hexdigest()[:8],16)).shuffle(group)
        n_val=max(1,round(len(group)*validation_fraction))
        if n_val>=len(group): raise ValueError(f'Need at least one training document for {source}')
        validation.extend(group[:n_val]);train.extend(group[n_val:])
        per_source[source]={'total':len(group),'training':len(group)-n_val,'validation':n_val}
    if {document.uid for document in train}&{document.uid for document in validation}: raise AssertionError('Split leakage')
    metadata={
        'created_utc':utc_now(),'seed':seed,'validation_fraction':validation_fraction,
        'strategy':'source-stratified deterministic document split','per_source':per_source,
        'dataset_fingerprint':dataset_fingerprint(documents),
        'training_ids':sorted(document.uid for document in train),
        'validation_ids':sorted(document.uid for document in validation),
    }
    return train,validation,metadata

def save_or_verify_split(path:Path,meta:Mapping[str,Any])->None:
    path=Path(path)
    if path.exists():
        old=json.loads(path.read_text(encoding='utf-8'))
        keys=('seed','validation_fraction','strategy','per_source','dataset_fingerprint','training_ids','validation_ids')
        if any(old.get(key)!=meta.get(key) for key in keys): raise RuntimeError('Existing data split differs; refusing overwrite')
        print('Verified existing split',path)
    else: atomic_write_json(path,dict(meta))

def validate_source_mixture(ratios:Mapping[str,float],sources:Iterable[str])->dict[str,float]:
    allowed=set(sources)
    if set(ratios)!=allowed: raise ValueError(f'Mixture keys {set(ratios)} must equal {allowed}')
    values={key:float(value) for key,value in ratios.items()}
    if any(not math.isfinite(value) or value<=0 for value in values.values()): raise ValueError('All source proportions must be finite and > 0')
    if not math.isclose(sum(values.values()),1.0,rel_tol=1e-6,abs_tol=1e-6): raise ValueError('Source proportions must sum to 1.0')
    return values

def source_sampling_indices(documents:Sequence[Document],ratios:Mapping[str,float],samples:int,seed:int)->list[int]:
    if samples<=0: raise ValueError('samples must be >0')
    ratios=validate_source_mixture(ratios,(document.source for document in documents))
    pools={source:[index for index,document in enumerate(documents) if document.source==source] for source in ratios}
    if any(not pool for pool in pools.values()): raise ValueError('Missing source documents')
    rng=random.Random(seed);sources=list(sorted(ratios));weights=[ratios[source] for source in sources]
    return [rng.choice(pools[source]) for source in rng.choices(sources,weights=weights,k=samples)]

def _stable_document_order(documents:Sequence[Document],seed:int)->list[Document]:
    return sorted(documents,key=lambda document:hashlib.sha256(f'{seed}:{document.uid}'.encode()).hexdigest())

def make_continuation_examples(documents:Sequence[Document],prompt_words:int=512,continuation_words:int=128,max_per_source:int=16,seed:int=781)->list[dict[str,str]]:
    result=[];needed=prompt_words+continuation_words
    for source in sorted({document.source for document in documents}):
        collected=0
        for document in _stable_document_order([value for value in documents if value.source==source],seed):
            if document.word_count<needed: continue
            units=document_text(document).split()
            result.append({'id':document.uid,'source':source,'prompt':' '.join(units[:prompt_words]),'reference':' '.join(units[prompt_words:needed])})
            collected+=1
            if collected>=max_per_source: break
    return result

def sampled_cross_split_overlap_audit(train:Sequence[Document],validation:Sequence[Document],seed:int,max_per_split:int=10_000)->dict[str,Any]:
    def signature(document:Document)->str:
        units=document_text(document).lower().split()
        sample=' '.join([*units[:128],*units[-128:]])
        return hashlib.sha256(sample.encode()).hexdigest()
    sampled_train=_stable_document_order(train,seed)[:max_per_split]
    sampled_val=_stable_document_order(validation,seed+1)[:max_per_split]
    train_signatures={signature(document):document.uid for document in sampled_train}
    overlaps=[]
    for document in sampled_val:
        value=signature(document)
        if value in train_signatures: overlaps.append({'training_uid':train_signatures[value],'validation_uid':document.uid})
    return {'method':'first+last 128 normalized whitespace words SHA-256','training_sample':len(sampled_train),'validation_sample':len(sampled_val),'overlaps':overlaps}


In [ ]:
def load_official_training_corpus(cfg:RunConfig,discovered:Mapping[str,Sequence[Path]]):
    if not cfg.document_id_field or not cfg.document_text_field: raise RuntimeError('Cell 10 must verify document fields first')
    if cfg.validation_fraction is None or cfg.source_mixing_ratios is None: raise RuntimeError('Set validation fraction and source mixture')
    ratios=validate_source_mixture(cfg.source_mixing_ratios,(Path(filename).stem.removeprefix('train_') for filename in cfg.train_files))
    all_docs=[];rejections={};provenance={};store_dir=DIRS['data']/'document_store'
    for filename in cfg.train_files:
        path=require_unique_file(discovered,filename);source=Path(filename).stem.removeprefix('train_')
        docs,bad=load_documents(path,source,cfg.document_id_field,cfg.document_text_field,store_dir=store_dir)
        all_docs.extend(docs);rejections[source]=bad
        provenance[source]={'file':filename,'bytes':path.stat().st_size,'sha256':sha256_file(path),'accepted':len(docs),'rejected':len(bad)}
    clean,dropped=deduplicate_exact_documents(all_docs)
    audit=corpus_audit(clean);audit['original_documents']=len(all_docs);audit['removed_duplicates']=len(dropped)
    audit['requested_sampling_ratios']=ratios;audit['sampling_unit']='document draws, not token percentages'
    train,validation,split=deterministic_document_split(clean,cfg.validation_fraction,cfg.seed)
    audit['training']=corpus_audit(train);audit['validation']=corpus_audit(validation)
    save_or_verify_split(DIRS['manifests']/'document_split.json',split)
    overlap=sampled_cross_split_overlap_audit(train,validation,cfg.seed)
    for name,obj in [
        ('dataset_provenance.json',provenance),('dataset_audit.json',audit),('rejected_records.json',rejections),
        ('deduplication_log.json',dropped),('source_sampling_config.json',{'ratios':ratios,'unit':'document draws'}),
        ('cross_split_overlap_audit.json',overlap),
    ]:
        manifest_path=DIRS['manifests']/name
        if manifest_path.exists() and json.loads(manifest_path.read_text(encoding='utf-8'))!=obj: raise RuntimeError(f'Persisted manifest changed: {manifest_path}')
        if not manifest_path.exists(): atomic_write_json(manifest_path,obj)
    examples=make_continuation_examples(validation,max_per_source=16,seed=cfg.seed)
    examples_path=DIRS['data']/'local_continuation_examples.json'
    if examples_path.exists() and json.loads(examples_path.read_text(encoding='utf-8'))!=examples: raise RuntimeError('Development examples changed')
    if not examples_path.exists(): atomic_write_json(examples_path,examples)
    atomic_write_json(DIRS['manifests']/'local_generation_protocol.json',{
        'selection':'up to 16 eligible documents per source in deterministic hash order','seed':cfg.seed,
        'prompt_words':512,'reference_words':128,'examples':len(examples),'ids':[row['id'] for row in examples],
    })
    register_run_config(cfg)
    return train,validation,examples,audit

if RESOLVED_SCHEMA is not None:
    TRAIN_DOCS,VAL_DOCS,LOCAL_DEV,DATA_AUDIT=load_official_training_corpus(CFG,DISCOVERED)
    print('Loaded disk-backed official corpus metadata:',DATA_AUDIT)
else:
    TRAIN_DOCS=VAL_DOCS=LOCAL_DEV=DATA_AUDIT=None
    print('Official corpus loading deferred; acquire/attach files and rerun Cells 9-14.')


## 5. Tokenizer persistence and compatibility

In [ ]:
def directory_fingerprint(path:Path)->str:
    path=Path(path);digest=hashlib.sha256()
    files=sorted(p for p in path.rglob('*') if p.is_file() and p.name!='tokenizer_metadata.json')
    if not files: raise ValueError(f'No tokenizer files found: {path}')
    for file in files: digest.update(file.relative_to(path).as_posix().encode()+bytes.fromhex(sha256_file(file)))
    return digest.hexdigest()
def tokenizer_encode(tok:Any,text:str)->list[int]:
    try: result=tok.encode(text,add_special_tokens=False)
    except TypeError: result=tok.encode(text)
    return list(result)
def tokenizer_decode(tok:Any,ids:Sequence[int])->str:
    try: return tok.decode(list(ids),skip_special_tokens=True)
    except TypeError: return tok.decode(list(ids))
def tokenizer_metadata(tok:Any,location:Path)->dict[str,Any]:
    probes=['The Court considered section 14 and dismissed the appeal.','§ 21—न्यायालय: AIR 2024 SC 17;  multiple   spaces\nand punctuation.']
    results=[]
    for probe in probes:
        ids=tokenizer_encode(tok,probe);decoded=tokenizer_decode(tok,ids)
        if not ids or decoded!=probe: raise ValueError(f'Tokenizer failed exact roundtrip: {probe!r} -> {decoded!r}')
        results.append({'input':probe,'ids':ids,'decoded':decoded})
    special={k:getattr(tok,k,None) for k in ('bos_token_id','eos_token_id','pad_token_id','unk_token_id')}
    if len({value for value in special.values() if value is not None})!=len([value for value in special.values() if value is not None]): raise ValueError('Special token IDs must be distinct')
    return {'created_utc':utc_now(),'vocab_size':len(tok),'special_token_ids':special,'fingerprint':directory_fingerprint(location),'round_trip_probes':results}
def register_tokenizer(tok:Any,destination:Path,save_function:Callable[[Any,Path],Any]|None=None)->dict[str,Any]:
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True);meta_path=destination/'tokenizer_metadata.json'
    if meta_path.exists():
        previous=json.loads(meta_path.read_text(encoding='utf-8'))
        if previous.get('fingerprint')!=directory_fingerprint(destination): raise RuntimeError('Tokenizer registered files have changed')
        if previous.get('vocab_size')!=len(tok): raise RuntimeError('Loaded tokenizer size differs')
        return previous
    if any(destination.iterdir()): raise RuntimeError('Nonempty tokenizer directory; choose explicit restore/adaptation')
    if save_function: save_function(tok,destination)
    elif hasattr(tok,'save_pretrained'): tok.save_pretrained(destination)
    else: raise TypeError('Course tokenizer requires an explicit save_function(tok,destination) adapter')
    meta=tokenizer_metadata(tok,destination);atomic_write_json(meta_path,meta);return meta
def tokenizer_token_audit(docs:Iterable[Document],tok:Any)->dict[str,Any]:
    by_source=Counter(); count=0
    for d in docs: by_source[d.source]+=len(tokenizer_encode(tok,document_text(d)));count+=1
    return {'documents':count,'tokens':sum(by_source.values()),'tokens_by_source':dict(by_source)}

COURSE_BPE_SOURCE ='"""Deterministic byte-level BPE tokenizer used by the Phase 1 notebook.\n\nThe implementation is deliberately small: it wraps the `tokenizers` BPE model,\nexposes the interface expected by the notebook, and persists every artifact\nneeded to recreate the token-to-ID mapping in a later Kaggle session.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\n\nSPECIAL_TOKENS = ("<pad>", "<unk>", "<bos>", "<eos>")\n\n\nclass CourseBPETokenizer:\n    """Minimal tokenizer adapter with a stable encode/decode/save contract."""\n\n    def __init__(self, backend):\n        self.backend = backend\n        self.pad_token_id = backend.token_to_id(SPECIAL_TOKENS[0])\n        self.unk_token_id = backend.token_to_id(SPECIAL_TOKENS[1])\n        self.bos_token_id = backend.token_to_id(SPECIAL_TOKENS[2])\n        self.eos_token_id = backend.token_to_id(SPECIAL_TOKENS[3])\n\n    def __len__(self) -> int:\n        return int(self.backend.get_vocab_size())\n\n    def encode(self, text: str, add_special_tokens: bool = False) -> list[int]:\n        if not isinstance(text, str):\n            raise TypeError("text must be a string")\n        ids = list(self.backend.encode(text, add_special_tokens=False).ids)\n        if add_special_tokens:\n            ids = [self.bos_token_id, *ids, self.eos_token_id]\n        return ids\n\n    def decode(self, ids: Sequence[int], skip_special_tokens: bool = True) -> str:\n        return self.backend.decode([int(value) for value in ids], skip_special_tokens=skip_special_tokens)\n\n    @classmethod\n    def train(\n        cls,\n        texts: Iterable[str],\n        *,\n        vocab_size: int = 32_000,\n        min_frequency: int = 2,\n    ) -> "CourseBPETokenizer":\n        try:\n            from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers\n        except ImportError as exc:\n            raise RuntimeError(\n                "The `tokenizers` package is required. Kaggle normally includes it; "\n                "otherwise install a pinned version before this cell."\n            ) from exc\n\n        if vocab_size < 1_024:\n            raise ValueError("vocab_size must be at least 1024")\n        if min_frequency < 1:\n            raise ValueError("min_frequency must be positive")\n\n        backend = Tokenizer(models.BPE(unk_token=SPECIAL_TOKENS[1]))\n        backend.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)\n        backend.decoder = decoders.ByteLevel()\n        trainer = trainers.BpeTrainer(\n            vocab_size=int(vocab_size),\n            min_frequency=int(min_frequency),\n            special_tokens=list(SPECIAL_TOKENS),\n            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),\n            show_progress=True,\n        )\n        backend.train_from_iterator((str(text) for text in texts), trainer=trainer)\n        tokenizer = cls(backend)\n        if len(tokenizer) != vocab_size:\n            raise RuntimeError(\n                f"BPE training produced {len(tokenizer)} tokens, expected exactly {vocab_size}. "\n                "Inspect the corpus and do not begin model training with a changing vocabulary."\n            )\n        return tokenizer\n\n    def save_pretrained(self, destination: str | Path) -> tuple[str, ...]:\n        destination = Path(destination)\n        destination.mkdir(parents=True, exist_ok=True)\n        tokenizer_json = destination / "tokenizer.json"\n        self.backend.save(str(tokenizer_json))\n        # Save the explicit BPE vocabulary and merge table as well as the unified\n        # tokenizer.json, so the learned mapping is auditable without code.\n        model_files = tuple(self.backend.model.save(str(destination), "course_bpe"))\n        config = {\n            "tokenizer_class": type(self).__name__,\n            "tokenizer_file": tokenizer_json.name,\n            "model_type": "byte_level_bpe",\n            "special_tokens": list(SPECIAL_TOKENS),\n            "special_token_ids": {\n                "pad_token_id": self.pad_token_id,\n                "unk_token_id": self.unk_token_id,\n                "bos_token_id": self.bos_token_id,\n                "eos_token_id": self.eos_token_id,\n            },\n            "vocab_size": len(self),\n        }\n        (destination / "tokenizer_config.json").write_text(\n            json.dumps(config, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8"\n        )\n        return (str(tokenizer_json), *model_files, str(destination / "tokenizer_config.json"))\n\n    @classmethod\n    def from_pretrained(cls, location: str | Path) -> "CourseBPETokenizer":\n        try:\n            from tokenizers import Tokenizer\n        except ImportError as exc:\n            raise RuntimeError("The `tokenizers` package is required to restore the BPE tokenizer") from exc\n\n        location = Path(location)\n        tokenizer_path = location / "tokenizer.json"\n        config_path = location / "tokenizer_config.json"\n        if not tokenizer_path.is_file() or not config_path.is_file():\n            raise FileNotFoundError(f"Incomplete tokenizer directory: {location}")\n        config = json.loads(config_path.read_text(encoding="utf-8"))\n        tokenizer = cls(Tokenizer.from_file(str(tokenizer_path)))\n        if len(tokenizer) != int(config["vocab_size"]):\n            raise RuntimeError("Saved tokenizer vocabulary size does not match tokenizer_config.json")\n        actual_ids = {\n            "pad_token_id": tokenizer.pad_token_id,\n            "unk_token_id": tokenizer.unk_token_id,\n            "bos_token_id": tokenizer.bos_token_id,\n            "eos_token_id": tokenizer.eos_token_id,\n        }\n        if actual_ids != config["special_token_ids"]:\n            raise RuntimeError("Saved tokenizer special-token mapping is inconsistent")\n        return tokenizer\n'

TOKENIZER_TRAINING_CONFIG={
    'type':'byte_level_bpe','vocab_size':32_000,'min_frequency':2,
    'training_documents':'TRAIN_DOCS only','special_tokens':['<pad>','<unk>','<bos>','<eos>'],
}
RUN_TOKENIZER_TRAINING=False

def install_python_source(path:Path,source:str)->None:
    path=Path(path)
    if path.exists() and path.read_text(encoding='utf-8')!=source and any(DIRS['checkpoints'].glob('*.pt')):
        raise RuntimeError(f'Refusing to change source after checkpoints exist: {path}')
    if not path.exists() or path.read_text(encoding='utf-8')!=source: atomic_write_bytes(path,source.encode('utf-8'))

def import_source_module(module_name:str,path:Path):
    spec=importlib.util.spec_from_file_location(module_name,path)
    if spec is None or spec.loader is None: raise ImportError(f'Cannot import {path}')
    module=importlib.util.module_from_spec(spec);sys.modules[module_name]=module;spec.loader.exec_module(module);return module

BPE_SOURCE_PATH=DIRS['model']/'course_bpe.py';install_python_source(BPE_SOURCE_PATH,COURSE_BPE_SOURCE)
course_bpe=import_source_module('cs781_course_bpe',BPE_SOURCE_PATH)
if (DIRS['tokenizer']/'tokenizer.json').exists():
    tokenizer=course_bpe.CourseBPETokenizer.from_pretrained(DIRS['tokenizer'])
elif TRAIN_DOCS and RUN_TOKENIZER_TRAINING:
    tokenizer=course_bpe.CourseBPETokenizer.train(
        (document_text(document) for document in TRAIN_DOCS),
        vocab_size=TOKENIZER_TRAINING_CONFIG['vocab_size'],min_frequency=TOKENIZER_TRAINING_CONFIG['min_frequency'],
    )
else: tokenizer=None

if tokenizer is not None:
    TOKENIZER_META=register_tokenizer(tokenizer,DIRS['tokenizer'])
    audit_docs=TRAIN_DOCS[:min(256,len(TRAIN_DOCS))] if TRAIN_DOCS else []
    TOKENIZER_AUDIT=tokenizer_token_audit(audit_docs,tokenizer) if audit_docs else None
    if TOKENIZER_AUDIT is not None:
        TOKENIZER_AUDIT['scope']='deterministic first 256 training documents (or all when fewer)'
        atomic_write_json(DIRS['manifests']/'tokenizer_audit.json',TOKENIZER_AUDIT)
    print('Restored/registered tokenizer:',TOKENIZER_META)
else:
    TOKENIZER_META=None;TOKENIZER_AUDIT=None
    print('No saved tokenizer found. Review TOKENIZER_TRAINING_CONFIG, then set RUN_TOKENIZER_TRAINING=True once.')


## 6. Decoder-only Transformer implementation and validation


In [ ]:
DECODER_MODEL_SOURCE = '"""Phase 1 decoder-only Transformer trained from scratch.\n\nThe architecture uses Pre-LayerNorm causal self-attention, learned absolute\npositions, a parameter-matched SwiGLU feed-forward intervention, and tied input\nand output token embeddings. No pretrained weights or external model code are\nused.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import asdict, dataclass\nfrom typing import Any, Mapping, Sequence\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass(frozen=True)\nclass ModelConfig:\n    vocab_size: int = 32_000\n    context_length: int = 1_024\n    hidden_size: int = 768\n    num_layers: int = 8\n    num_heads: int = 12\n    ffn_hidden_size: int = 2_048\n    dropout: float = 0.1\n    layer_norm_epsilon: float = 1e-5\n    tie_word_embeddings: bool = True\n    activation: str = "swiglu"\n    architecture: str = "phase1_preln_decoder"\n\n    def __post_init__(self) -> None:\n        if self.vocab_size <= 0 or self.context_length <= 1:\n            raise ValueError("vocab_size and context_length must be positive")\n        if self.hidden_size % self.num_heads:\n            raise ValueError("hidden_size must be divisible by num_heads")\n        if self.num_layers <= 0 or self.ffn_hidden_size <= 0:\n            raise ValueError("num_layers and ffn_hidden_size must be positive")\n        if not 0.0 <= self.dropout < 1.0:\n            raise ValueError("dropout must be in [0, 1)")\n        if self.activation != "swiglu":\n            raise ValueError("This submission implements the recorded SwiGLU intervention")\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n    @classmethod\n    def from_dict(cls, values: Mapping[str, Any]) -> "ModelConfig":\n        return cls(**dict(values))\n\n\nclass CausalSelfAttention(nn.Module):\n    def __init__(self, config: ModelConfig):\n        super().__init__()\n        self.num_heads = config.num_heads\n        self.head_dim = config.hidden_size // config.num_heads\n        self.dropout = float(config.dropout)\n        self.qkv = nn.Linear(config.hidden_size, 3 * config.hidden_size, bias=False)\n        self.output = nn.Linear(config.hidden_size, config.hidden_size, bias=False)\n\n    def forward(\n        self,\n        hidden: torch.Tensor,\n        past_key_value: tuple[torch.Tensor, torch.Tensor] | None = None,\n        use_cache: bool = False,\n    ) -> torch.Tensor | tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:\n        batch, sequence, width = hidden.shape\n        qkv = self.qkv(hidden)\n        query, key, value = qkv.chunk(3, dim=-1)\n        query = query.view(batch, sequence, self.num_heads, self.head_dim).transpose(1, 2)\n        key = key.view(batch, sequence, self.num_heads, self.head_dim).transpose(1, 2)\n        value = value.view(batch, sequence, self.num_heads, self.head_dim).transpose(1, 2)\n        past_length = 0\n        if past_key_value is not None:\n            past_key, past_value = past_key_value\n            if past_key.shape[:2] != key.shape[:2] or past_key.shape[3] != key.shape[3]:\n                raise ValueError("Incompatible attention cache")\n            past_length = past_key.shape[2]\n            key = torch.cat((past_key, key), dim=2)\n            value = torch.cat((past_value, value), dim=2)\n        # PyTorch\'s fused implementation evaluates softmax(QK^T/sqrt(d_k) + M)V\n        # with the strict upper triangle masked when is_causal=True.\n        if past_length:\n            query_positions = past_length + torch.arange(sequence, device=hidden.device)\n            key_positions = torch.arange(key.shape[2], device=hidden.device)\n            allowed = key_positions.unsqueeze(0) <= query_positions.unsqueeze(1)\n            attended = F.scaled_dot_product_attention(\n                query, key, value, attn_mask=allowed,\n                dropout_p=self.dropout if self.training else 0.0, is_causal=False,\n            )\n        else:\n            attended = F.scaled_dot_product_attention(\n                query, key, value, attn_mask=None,\n                dropout_p=self.dropout if self.training else 0.0, is_causal=True,\n            )\n        attended = attended.transpose(1, 2).contiguous().view(batch, sequence, width)\n        result = self.output(attended)\n        return (result, (key, value)) if use_cache else result\n\n\nclass SwiGLUFeedForward(nn.Module):\n    def __init__(self, config: ModelConfig):\n        super().__init__()\n        self.gate = nn.Linear(config.hidden_size, config.ffn_hidden_size, bias=False)\n        self.up = nn.Linear(config.hidden_size, config.ffn_hidden_size, bias=False)\n        self.down = nn.Linear(config.ffn_hidden_size, config.hidden_size, bias=False)\n        self.dropout = nn.Dropout(config.dropout)\n\n    def forward(self, hidden: torch.Tensor) -> torch.Tensor:\n        return self.dropout(self.down(F.silu(self.gate(hidden)) * self.up(hidden)))\n\n\nclass DecoderBlock(nn.Module):\n    def __init__(self, config: ModelConfig):\n        super().__init__()\n        self.attention_norm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_epsilon)\n        self.attention = CausalSelfAttention(config)\n        self.attention_dropout = nn.Dropout(config.dropout)\n        self.ffn_norm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_epsilon)\n        self.ffn = SwiGLUFeedForward(config)\n\n    def forward(\n        self,\n        hidden: torch.Tensor,\n        past_key_value: tuple[torch.Tensor, torch.Tensor] | None = None,\n        use_cache: bool = False,\n    ) -> torch.Tensor | tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:\n        attention_result = self.attention(\n            self.attention_norm(hidden), past_key_value=past_key_value, use_cache=use_cache\n        )\n        if use_cache:\n            attention_output, present = attention_result\n        else:\n            attention_output, present = attention_result, None\n        hidden = hidden + self.attention_dropout(attention_output)\n        hidden = hidden + self.ffn(self.ffn_norm(hidden))\n        return (hidden, present) if use_cache else hidden\n\n\nclass CausalDecoderLM(nn.Module):\n    def __init__(self, config: ModelConfig):\n        super().__init__()\n        self.config = config\n        self.token_embedding = nn.Embedding(config.vocab_size, config.hidden_size)\n        self.position_embedding = nn.Embedding(config.context_length, config.hidden_size)\n        self.embedding_dropout = nn.Dropout(config.dropout)\n        self.blocks = nn.ModuleList(DecoderBlock(config) for _ in range(config.num_layers))\n        self.final_norm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_epsilon)\n        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)\n        self.gradient_checkpointing = False\n        self.apply(self._initialize)\n        # GPT-style residual projection scaling keeps the initial residual stream stable.\n        residual_std = 0.02 / math.sqrt(2 * config.num_layers)\n        for block in self.blocks:\n            nn.init.normal_(block.attention.output.weight, mean=0.0, std=residual_std)\n            nn.init.normal_(block.ffn.down.weight, mean=0.0, std=residual_std)\n        if config.tie_word_embeddings:\n            self.lm_head.weight = self.token_embedding.weight\n\n    @staticmethod\n    def _initialize(module: nn.Module) -> None:\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n        if isinstance(module, nn.Linear) and module.bias is not None:\n            nn.init.zeros_(module.bias)\n\n    def set_gradient_checkpointing(self, enabled: bool = True) -> None:\n        self.gradient_checkpointing = bool(enabled)\n\n    def get_config(self) -> dict[str, Any]:\n        return self.config.to_dict()\n\n    def forward(\n        self,\n        input_ids: torch.Tensor,\n        past_key_values: Sequence[tuple[torch.Tensor, torch.Tensor]] | None = None,\n        use_cache: bool = False,\n    ) -> torch.Tensor | tuple[torch.Tensor, tuple[tuple[torch.Tensor, torch.Tensor], ...]]:\n        if input_ids.ndim != 2 or input_ids.dtype != torch.long:\n            raise TypeError("input_ids must be a LongTensor shaped [batch, sequence]")\n        _, sequence = input_ids.shape\n        if past_key_values is not None and len(past_key_values) != len(self.blocks):\n            raise ValueError("past_key_values must contain one entry per decoder block")\n        past_length = 0 if past_key_values is None else int(past_key_values[0][0].shape[2])\n        if sequence + past_length > self.config.context_length:\n            raise ValueError(\n                f"cached sequence length {sequence + past_length} exceeds context length {self.config.context_length}"\n            )\n        if input_ids.numel() and (int(input_ids.min()) < 0 or int(input_ids.max()) >= self.config.vocab_size):\n            raise ValueError("input_ids contain a token outside the configured vocabulary")\n        positions = torch.arange(past_length, past_length + sequence, device=input_ids.device)\n        hidden = self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :])\n        presents = []\n        for index, block in enumerate(self.blocks):\n            past = None if past_key_values is None else past_key_values[index]\n            if self.gradient_checkpointing and self.training and not use_cache:\n                hidden = checkpoint(block, hidden, use_reentrant=False)\n            else:\n                block_result = block(hidden, past_key_value=past, use_cache=use_cache)\n                if use_cache:\n                    hidden, present = block_result\n                    presents.append(present)\n                else:\n                    hidden = block_result\n        logits = self.lm_head(self.final_norm(hidden))\n        return (logits, tuple(presents)) if use_cache else logits\n\n\ndef build_model(config: Mapping[str, Any] | ModelConfig) -> CausalDecoderLM:\n    if not isinstance(config, ModelConfig):\n        config = ModelConfig.from_dict(config)\n    return CausalDecoderLM(config)\n'

MODEL_SOURCE_PATH=DIRS['model']/'decoder_lm.py';install_python_source(MODEL_SOURCE_PATH,DECODER_MODEL_SOURCE)
decoder_lm=import_source_module('cs781_decoder_lm',MODEL_SOURCE_PATH)
MODEL_CONFIG={
    'vocab_size':int(TOKENIZER_META['vocab_size'] if TOKENIZER_META else TOKENIZER_TRAINING_CONFIG['vocab_size']),
    'context_length':1024,'hidden_size':768,'num_layers':8,'num_heads':12,'ffn_hidden_size':2048,
    'dropout':0.1,'layer_norm_epsilon':1e-5,'tie_word_embeddings':True,
    'activation':'swiglu','architecture':'phase1_preln_decoder',
}
RECONSTRUCT_MODEL=decoder_lm.build_model;MODEL=RECONSTRUCT_MODEL(MODEL_CONFIG)

def count_parameters(model:torch.nn.Module)->dict[str,int]:
    return {'total':sum(parameter.numel() for parameter in model.parameters()),'trainable':sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)}
def _logits(output:Any)->torch.Tensor:
    return output.logits if hasattr(output,'logits') else output[0] if isinstance(output,(tuple,list)) else output
def validate_model_contract(model:torch.nn.Module,vocab_size:int,device:str,sequence_length:int=16)->dict[str,Any]:
    model=model.to(device);model.train();model.zero_grad(set_to_none=True)
    input_ids=torch.randint(0,vocab_size,(2,sequence_length),dtype=torch.long,device=device);logits=_logits(model(input_ids))
    expected=(2,sequence_length,vocab_size)
    if tuple(logits.shape)!=expected: raise ValueError(f'Expected logits shape {expected}, got {tuple(logits.shape)}')
    if not torch.isfinite(logits).all(): raise FloatingPointError('Non-finite logits')
    loss=torch.nn.functional.cross_entropy(logits[:,:-1].reshape(-1,vocab_size),input_ids[:,1:].reshape(-1));loss.backward()
    gradients=[parameter.grad for parameter in model.parameters() if parameter.requires_grad and parameter.grad is not None]
    if not gradients or not all(torch.isfinite(gradient).all().item() for gradient in gradients): raise FloatingPointError('Missing/non-finite gradients')
    counts=count_parameters(model)
    if not 50_000_000<=counts['total']<=300_000_000: raise ValueError(f"Official parameter bound violated: {counts['total']:,}")
    model.zero_grad(set_to_none=True)
    return {'logits_shape':list(logits.shape),'loss':float(loss.detach().cpu()),**counts,'device':device}

if TOKENIZER_META is not None:
    MODEL_DEVICE='cuda:0' if torch.cuda.is_available() else 'cpu';MODEL_REPORT=validate_model_contract(MODEL,TOKENIZER_META['vocab_size'],MODEL_DEVICE)
    atomic_write_json(DIRS['manifests']/'model_validation.json',MODEL_REPORT);MODEL.to('cpu')
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('Model validation:',MODEL_REPORT)
else:
    MODEL_REPORT={'status':'DEFERRED_UNTIL_TOKENIZER_REGISTRATION',**count_parameters(MODEL)};print(MODEL_REPORT)


## 7. Stateful sampling, histories, checkpoints, and progress

In [ ]:
class StatefulRandomSampler(Sampler[int]):
    # Cursor is updated when DataLoader requests indices, possibly ahead of consumed batches if num_workers>0.
    # For exact checkpoint continuation use num_workers=0 or save a committed (consumed) batch cursor.
    def __init__(self,data_source:Sequence[Any],seed:int,epoch:int=0,cursor:int=0):
        self.length=len(data_source);self.seed=int(seed);self.epoch=int(epoch);self.cursor=int(cursor)
    def _order(self)->list[int]:
        return torch.randperm(self.length,generator=torch.Generator().manual_seed(self.seed+self.epoch)).tolist()
    def __iter__(self):
        for position in range(self.cursor,self.length):
            self.cursor=position+1
            yield self._order_cache[position] if hasattr(self,'_order_cache') else self._get_order_item(position)
    def _get_order_item(self,position):
        if not hasattr(self,'_order_cache'): self._order_cache=self._order()
        return self._order_cache[position]
    def __len__(self): return max(0,self.length-self.cursor)
    def next_epoch(self): self.epoch+=1;self.cursor=0;self.__dict__.pop('_order_cache',None)
    def state_dict(self): return {'length':self.length,'seed':self.seed,'epoch':self.epoch,'cursor':self.cursor}
    def load_state_dict(self,state:Mapping[str,Any]):
        if int(state['length'])!=self.length or int(state['seed'])!=self.seed: raise ValueError('Incompatible sampler')
        cursor=int(state['cursor'])
        if not 0<=cursor<=self.length: raise ValueError('Sampler cursor outside dataset')
        self.epoch=int(state['epoch']);self.cursor=cursor;self.__dict__.pop('_order_cache',None)

class JsonlHistory:
    def __init__(self,path:Path): self.path=Path(path);self.path.parent.mkdir(parents=True,exist_ok=True)
    def records(self)->list[dict[str,Any]]:
        if not self.path.exists(): return []
        lines=self.path.read_text(encoding='utf-8').splitlines();rows=[]
        for i,line in enumerate(lines):
            if not line.strip():continue
            try: rows.append(json.loads(line))
            except json.JSONDecodeError:
                if i==len(lines)-1: raise RuntimeError('Incomplete last history line. Restore last checkpoint/history or explicitly repair; do not silently discard')
                raise
        steps=[r['step'] for r in rows]
        if len(steps)!=len(set(steps)) or steps!=sorted(steps): raise ValueError('Duplicate or nonmonotonic history steps')
        return rows
    def append(self,record:Mapping[str,Any]):
        existing=self.records()
        if existing and int(record['step'])<=int(existing[-1]['step']): raise ValueError('Refusing duplicated/nonmonotonic history step')
        with self.path.open('ab') as h:
            h.write((json.dumps(dict(record),ensure_ascii=False)+'\n').encode());h.flush();os.fsync(h.fileno())

In [ ]:
CHECKPOINT_VERSION=2
def capture_rng_state(device:str|None=None)->dict[str,Any]:
    gpu=None
    if device and str(device).startswith('cuda') and torch.cuda.is_available():
        gpu={'index':torch.device(device).index or 0,'state':torch.cuda.get_rng_state(torch.device(device))}
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch_cpu':torch.get_rng_state(),'torch_cuda_device':gpu}
def restore_rng_state(state:Mapping[str,Any])->None:
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch_cpu'])
    gpu=state.get('torch_cuda_device')
    if gpu is not None:
        if not torch.cuda.is_available() or gpu['index']>=torch.cuda.device_count(): raise RuntimeError('Saved CUDA RNG device unavailable')
        torch.cuda.set_rng_state(gpu['state'],device=gpu['index'])
def save_checkpoint_atomic(path:Path,*,model:torch.nn.Module,optimizer:torch.optim.Optimizer,scheduler:Any,scaler:Any,sampler:StatefulRandomSampler,progress:Mapping[str,Any],model_config:Mapping[str,Any],tokenizer_fingerprint:str,dataset_fingerprint_value:str,training_config:Mapping[str,Any],history_path:Path,validation_history:Sequence[Mapping[str,Any]],experiment_id:str,device:str|None=None)->dict[str,Any]:
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    hist=sha256_file(history_path) if Path(history_path).exists() else None
    state={'checkpoint_version':CHECKPOINT_VERSION,'experiment_id':experiment_id,'created_utc':utc_now(),'model_state':model.state_dict(),'optimizer_state':optimizer.state_dict(),'scheduler_state':scheduler.state_dict() if scheduler is not None else None,'scaler_state':scaler.state_dict() if scaler is not None else None,'sampler_state':sampler.state_dict(),'rng_state':capture_rng_state(device),'progress':dict(progress),'model_config':dict(model_config),'tokenizer_fingerprint':tokenizer_fingerprint,'dataset_fingerprint':dataset_fingerprint_value,'training_config':dict(training_config),'history_sha256':hist,'validation_history':list(validation_history)}
    with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as h: tmp=Path(h.name)
    try:
        torch.save(state,tmp)
        check=torch.load(tmp,map_location='cpu',weights_only=False)
        if check.get('checkpoint_version')!=CHECKPOINT_VERSION or check.get('progress')!=dict(progress): raise RuntimeError('Checkpoint readback failed')
        digest=sha256_file(tmp)
        if path.exists():
            previous=path.with_name('previous_known_good.pt')
            with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as h: prev_tmp=Path(h.name)
            try:
                shutil.copyfile(path,prev_tmp)
                torch.load(prev_tmp,map_location='cpu',weights_only=False)
                os.replace(prev_tmp,previous)
            finally: prev_tmp.unlink(missing_ok=True)
        os.replace(tmp,path)
        return {'path':str(path),'sha256':digest,'step':progress.get('step')}
    finally: tmp.unlink(missing_ok=True)

In [ ]:
def restore_checkpoint(path:Path,*,model:torch.nn.Module,optimizer:torch.optim.Optimizer,sampler:StatefulRandomSampler,expected_tokenizer_fingerprint:str,expected_dataset_fingerprint:str,scheduler:Any=None,scaler:Any=None,expected_experiment_id:str|None=None,expected_model_config:Mapping[str,Any]|None=None,history_path:Path|None=None,device:str|None=None)->dict[str,Any]:
    state=torch.load(Path(path),map_location='cpu',weights_only=False) # Only load your own trusted checkpoint files.
    if state.get('checkpoint_version')!=CHECKPOINT_VERSION: raise ValueError('Checkpoint schema mismatch')
    if expected_experiment_id is not None and state['experiment_id']!=expected_experiment_id: raise ValueError('Experiment ID mismatch')
    if state['tokenizer_fingerprint']!=expected_tokenizer_fingerprint: raise ValueError('Tokenizer fingerprint mismatch')
    if state['dataset_fingerprint']!=expected_dataset_fingerprint: raise ValueError('Dataset fingerprint mismatch')
    if expected_model_config is not None and dict(expected_model_config)!=state['model_config']: raise ValueError('Architecture config mismatch')
    if history_path is not None:
        actual=sha256_file(history_path) if Path(history_path).exists() else None
        if actual!=state['history_sha256']: raise ValueError('History file differs from checkpoint; restore matching history or explicitly reconcile after crash')
    if (scheduler is None)!=(state['scheduler_state'] is None): raise ValueError('Scheduler state incompatibility')
    if (scaler is None)!=(state['scaler_state'] is None): raise ValueError('AMP scaler state incompatibility')
    model.load_state_dict(state['model_state'],strict=True);optimizer.load_state_dict(state['optimizer_state'])
    if device is not None:
        for opt_state in optimizer.state.values():
            for k,v in opt_state.items():
                if isinstance(v,torch.Tensor): opt_state[k]=v.to(device)
    if scheduler is not None: scheduler.load_state_dict(state['scheduler_state'])
    if scaler is not None: scaler.load_state_dict(state['scaler_state'])
    sampler.load_state_dict(state['sampler_state']);restore_rng_state(state['rng_state'])
    return state

@dataclass
class ProgressState:
    experiment_id:str
    device:str
    total_steps:int
    step:int=0
    epoch:int=0
    tokens_processed:int=0
    cumulative_seconds:float=0.0
    last_checkpoint_step:int=0

class ProgressReporter:
    def __init__(self,state:ProgressState):
        self.state=state;self.start=time.perf_counter();self.base_cumulative=state.cumulative_seconds;self.last=self.start;self.last_step=state.step;self.last_tokens=state.tokens_processed
        self.bar=tqdm(total=state.total_steps,initial=state.step,desc=f'{state.experiment_id} [{state.device}]')
    def update(self,*,step:int,epoch:int,loss:float,learning_rate:float,tokens_processed:int,gradient_norm:float|None):
        now=time.perf_counter();dt=max(now-self.last,1e-9);ds=step-self.last_step;dtoken=tokens_processed-self.last_tokens
        if ds<0 or dtoken<0: raise ValueError('Progress cannot go backwards')
        cumulative=self.base_cumulative+(now-self.start);rate=ds/dt
        self.bar.update(ds);self.bar.set_postfix({'epoch':epoch,'loss':f'{loss:.4f}','lr':f'{learning_rate:.3g}','tok':tokens_processed,'tok/s':f'{dtoken/dt:.0f}','step/s':f'{rate:.2f}','grad':'NA' if gradient_norm is None else f'{gradient_norm:.3f}','elapsed_s':f'{cumulative:.0f}','eta_s':f'{max(0,self.state.total_steps-step)/rate:.0f}' if rate>0 else 'inf','ckpt':self.state.last_checkpoint_step})
        self.state.step=step;self.state.epoch=epoch;self.state.tokens_processed=tokens_processed;self.state.cumulative_seconds=cumulative;self.last=now;self.last_step=step;self.last_tokens=tokens_processed
    def close(self): self.bar.close()

def reconcile_history_to_checkpoint(checkpoint_path:Path,history_path:Path,*,apply:bool=False)->dict[str,Any]:
    checkpoint=torch.load(Path(checkpoint_path),map_location='cpu',weights_only=False)
    expected=checkpoint.get('history_sha256');history_path=Path(history_path)
    payload=history_path.read_bytes() if history_path.exists() else b''
    if (hashlib.sha256(payload).hexdigest() if payload else None)==expected:
        return {'status':'already_matching','kept_bytes':len(payload),'discarded_bytes':0}
    boundaries=[0];boundaries.extend(index+1 for index,value in enumerate(payload) if value==10)
    match=None
    for boundary in boundaries:
        prefix=payload[:boundary];digest=hashlib.sha256(prefix).hexdigest() if prefix else None
        if digest==expected: match=boundary
    if match is None: raise RuntimeError('History is not a strict extension of the checkpoint history; restore a matching recovery archive')
    report={'status':'rollback_available','kept_bytes':match,'discarded_bytes':len(payload)-match,'checkpoint_step':checkpoint['progress']['step']}
    if apply:
        backup=history_path.with_name(f"{history_path.name}.uncommitted.after_step_{checkpoint['progress']['step']}.{int(time.time())}")
        atomic_write_bytes(backup,payload);atomic_write_bytes(history_path,payload[:match]);report.update(status='rolled_back',backup=str(backup))
    return report


## 8. Recovery/bootstrap discovery

In [ ]:
def validate_zip(path:Path)->dict[str,Any]:
    with zipfile.ZipFile(path,'r') as z:
        bad=z.testzip()
        if bad is not None: raise zipfile.BadZipFile(f'CRC failure: {bad}')
        return {'path':str(path),'members':z.namelist(),'sha256':sha256_file(path)}
def discover_recovery_archives(input_root:Path)->list[Path]:
    return sorted(Path(input_root).rglob('phase1_submission01_recovery.zip')) if Path(input_root).exists() else []
RECOVERY_CANDIDATES=discover_recovery_archives(Path(CFG.input_root))
print('Recovery candidates:',[str(p) for p in RECOVERY_CANDIDATES])
if len(RECOVERY_CANDIDATES)>1: print('Multiple recovery archives found; choose explicit path.')
def verify_recovery_archive(path:Path)->dict[str,Any]:
    import stat
    result=validate_zip(path)
    with zipfile.ZipFile(path) as z:
        names=z.namelist()
        if len(names)!=len(set(names)): raise ValueError('Duplicate ZIP member names')
        if 'manifests/recovery_integrity.json' not in names: raise ValueError('No recovery member SHA manifest')
        manifest=json.loads(z.read('manifests/recovery_integrity.json'))
        expected={r['path']:r for r in manifest['files']}
        actual=set(n for n in names if not n.endswith('/') and n!='manifests/recovery_integrity.json')
        if set(expected)!=actual: raise ValueError(f'Manifest/archive mismatch: missing={set(expected)-actual}, extras={actual-set(expected)}')
        for member in z.infolist():
            p=Path(member.filename)
            if p.is_absolute() or '..' in p.parts or not member.filename or member.filename.startswith('\\'): raise ValueError('Unsafe member path')
            if stat.S_ISLNK(member.external_attr>>16): raise ValueError('Symlink in archive is forbidden')
        for name,item in expected.items():
            if hashlib.sha256(z.read(name)).hexdigest()!=item['sha256']: raise ValueError(f'SHA mismatch: {name}')
            if z.getinfo(name).file_size!=item['bytes']: raise ValueError(f'Size mismatch: {name}')
    return result
def restore_recovery_archive(archive_path:Path,destination:Path,*,allow_existing_identical:bool=True)->None:
    verify_recovery_archive(archive_path)
    dest=Path(destination).resolve();dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive_path) as z:
        # Preflight every path and conflict BEFORE extracting anything.
        for member in z.infolist():
            target=(dest/member.filename).resolve()
            if dest not in target.parents and target!=dest: raise ValueError(f'Unsafe path: {member.filename}')
            if member.is_dir(): continue
            if target.exists():
                if not target.is_file(): raise RuntimeError(f'Restore path is not a file: {target}')
                if not allow_existing_identical or hashlib.sha256(z.read(member)).hexdigest()!=sha256_file(target):
                    raise RuntimeError(f'Restore conflict; refusing to overwrite existing file: {target}')
        for member in z.infolist():
            if member.is_dir(): continue
            target=dest/member.filename
            if target.exists():continue
            atomic_write_bytes(target,z.read(member))

## 9. Disposable smoke tests before full training

In [ ]:
def run_infrastructure_smoke_tests() -> dict[str, str]:
    results: dict[str, str] = {}
    with tempfile.TemporaryDirectory() as temporary_name:
        temporary = Path(temporary_name)
        fixture = [
            {"id": "a", "text": " ".join(["alpha"] * 700)},
            {"id": "b", "text": " ".join(["beta"] * 700)},
            {"id": "c", "text": " ".join(["gamma"] * 700)},
        ]
        dataset_path = temporary / "fixture.json"
        atomic_write_json(dataset_path, fixture)
        docs, rejected = load_documents(dataset_path, "fixture", "id", "text")
        assert len(docs) == 3 and not rejected
        results["dataset_loading_and_schema"] = "PASS"
        train, validation, split = deterministic_document_split(docs, 1 / 3, 781)
        assert not ({doc.uid for doc in train} & {doc.uid for doc in validation})
        save_or_verify_split(temporary / "split.json", split)
        results["document_split_and_leakage"] = "PASS"
        examples = make_continuation_examples(validation)
        assert len(examples) == 1 and len(examples[0]["prompt"].split()) == 512 and len(examples[0]["reference"].split()) == 128
        results["local_continuation_examples"] = "PASS"
        sampler = StatefulRandomSampler(range(10), seed=9)
        iterator = iter(sampler)
        prefix = [next(iterator) for _ in range(4)]
        state = sampler.state_dict()
        resumed = StatefulRandomSampler(range(10), seed=9)
        resumed.load_state_dict(state)
        suffix = list(resumed)
        assert len(prefix + suffix) == 10 and len(set(prefix + suffix)) == 10
        results["sampler_continuation"] = "PASS"
        history = JsonlHistory(temporary / "history.jsonl")
        history.append({"step": 1, "loss": 1.0})
        assert history.records() == [{"step": 1, "loss": 1.0}]
        results["history_continuity"] = "PASS"
        dummy_prediction = {"1": {"text": " ".join(["word"] * 128)}}
        prediction_path = temporary / "prediction.json"
        atomic_write_json(prediction_path, dummy_prediction)
        zip_path = temporary / "submission.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
            archive.write(prediction_path, "prediction.json")
        assert validate_zip(zip_path)["members"] == ["prediction.json"]
        results["zip_integrity"] = "PASS"
    return results

SMOKE_RESULTS = run_infrastructure_smoke_tests()
print(json.dumps(SMOKE_RESULTS, indent=2))

In [ ]:
def validate_causal_mask(model:torch.nn.Module,vocab_size:int,device:str)->None:
    model.eval();first=torch.randint(0,vocab_size,(1,12),dtype=torch.long,device=device);second=first.clone()
    second[:,7:]=torch.randint(0,vocab_size,(1,5),dtype=torch.long,device=device)
    with torch.no_grad(): first_logits=_logits(model(first))[:,:7];second_logits=_logits(model(second))[:,:7]
    if not torch.allclose(first_logits,second_logits,atol=1e-5,rtol=1e-5): raise AssertionError('Causal masking is broken')

def validate_kv_cache(model:torch.nn.Module,vocab_size:int,device:str)->None:
    model.eval();ids=torch.randint(0,vocab_size,(1,10),dtype=torch.long,device=device)
    with torch.no_grad():
        full=_logits(model(ids))
        first=model(ids[:,:6],use_cache=True);past=first[1];pieces=[first[0]]
        for index in range(6,ids.shape[1]):
            current=model(ids[:,index:index+1],past_key_values=past,use_cache=True);pieces.append(current[0]);past=current[1]
        cached=torch.cat(pieces,dim=1)
    if not torch.allclose(full,cached,atol=2e-5,rtol=2e-5): raise AssertionError('KV-cache logits differ from full forward logits')

def run_student_integration_smoke_test(model,tokenizer,model_config,reconstruct_model,device=None):
    device=device or ('cuda:0' if torch.cuda.is_available() else 'cpu');meta=tokenizer_metadata(tokenizer,DIRS['tokenizer'])
    report=validate_model_contract(model,meta['vocab_size'],device);validate_causal_mask(model,meta['vocab_size'],device);report['causal_future_leakage']='PASS'
    validate_kv_cache(model,meta['vocab_size'],device);report['kv_cache_equivalence']='PASS'
    probe_ids=tokenizer_encode(tokenizer,'A constitutional court considered the appeal.')
    if not probe_ids or min(probe_ids)<0 or max(probe_ids)>=meta['vocab_size']: raise AssertionError('Tokenizer emitted an out-of-vocabulary ID')
    report['token_id_range']='PASS'
    try: model(torch.zeros((1,int(model_config['context_length'])+1),dtype=torch.long,device=device));raise AssertionError('Context limit not enforced')
    except ValueError: report['context_limit']='PASS'
    clone=reconstruct_model(model_config);clone.load_state_dict(model.state_dict(),strict=True);report['reconstruction']='PASS';del clone
    model.to('cpu')
    if str(device).startswith('cuda'): torch.cuda.empty_cache()
    atomic_write_json(DIRS['manifests']/'student_integration_smoke_test.json',report);return report

if TOKENIZER_META is not None:
    INTEGRATION_SMOKE_RESULTS=run_student_integration_smoke_test(MODEL,tokenizer,MODEL_CONFIG,RECONSTRUCT_MODEL);print(json.dumps(INTEGRATION_SMOKE_RESULTS,indent=2))
else: INTEGRATION_SMOKE_RESULTS=None;print('Model/tokenizer smoke test deferred until tokenizer registration.')


## 10. Mixed-source causal-language-model training


In [ ]:
TRAINING_CONFIG={
    'experiment_id':CFG.experiment_id,'objective':'valid-token-weighted shifted next-token cross-entropy',
    'optimizer':'AdamW','learning_rate':3e-4,'betas':[0.9,0.95],'epsilon':1e-8,'weight_decay':0.1,
    'schedule':'linear warmup then cosine decay','warmup_steps':500,'min_lr_ratio':0.10,
    'batch_size':1,'gradient_accumulation_steps':8,'sequence_length':1024,'max_steps':12_000,
    'samples_per_epoch':65_536,'precision_policy':'bf16 if supported, otherwise fp16 on CUDA, fp32 on CPU',
    'gradient_clip_norm':1.0,'gradient_checkpointing':True,'validation_interval':250,
    'validation_sequences_per_source':8,'checkpoint_interval':250,'recovery_interval':1000,
    'history_interval':1,'num_workers':0,'source_mixture':dict(CFG.source_mixing_ratios or {}),
    'source_mixture_unit':'document draws (resampled reproducibly every epoch), not token percentages',
    'seed':CFG.seed,'auto_reconcile_strict_history_extension':True,
    'context_policy':'train at the full 1024-token model context; generation keeps the latest 1024 BPE tokens',
}

class MixedDocumentCausalDataset(Dataset):
    def __init__(self,documents,tokenizer,ratios,samples,sequence_length,seed,cache_documents=128):
        self.documents=list(documents);self.tokenizer=tokenizer;self.ratios=dict(ratios);self.samples=int(samples)
        self.sequence_length=int(sequence_length);self.seed=int(seed);self.epoch=0;self.cache_documents=int(cache_documents);self.cache={}
        self.pad_id=int(getattr(tokenizer,'pad_token_id',0) or 0);self.eos_id=int(getattr(tokenizer,'eos_token_id',self.pad_id) or self.pad_id)
        self.unk_id=int(getattr(tokenizer,'unk_token_id',self.eos_id) or self.eos_id);self._resample()
    def _resample(self): self.document_indices=source_sampling_indices(self.documents,self.ratios,self.samples,self.seed+self.epoch)
    def __len__(self): return self.samples
    def set_epoch(self,epoch:int): self.epoch=int(epoch);self._resample()
    def _tokens(self,document_index:int)->list[int]:
        if document_index not in self.cache:
            if len(self.cache)>=self.cache_documents: self.cache.pop(next(iter(self.cache)))
            self.cache[document_index]=tokenizer_encode(self.tokenizer,document_text(self.documents[document_index]))
        return self.cache[document_index]
    def __getitem__(self,index:int):
        document_index=self.document_indices[int(index)];ids=self._tokens(document_index) or [self.unk_id];needed=self.sequence_length+1
        if len(ids)>=needed:
            rng=random.Random(self.seed+self.epoch*len(self)+int(index));start=rng.randrange(0,len(ids)-needed+1)
            sequence=list(ids[start:start+needed]);valid_targets=self.sequence_length
        else:
            sequence=[*ids,self.eos_id];valid_targets=max(1,len(sequence)-1);sequence.extend([self.pad_id]*(needed-len(sequence)))
            sequence=sequence[:needed];valid_targets=min(valid_targets,self.sequence_length)
        labels=torch.tensor(sequence[1:],dtype=torch.long)
        if valid_targets<self.sequence_length: labels[valid_targets:]=-100
        return {'input_ids':torch.tensor(sequence[:-1],dtype=torch.long),'labels':labels,'document_index':document_index}

def build_fixed_validation_rows(documents,tokenizer,sequence_length,sequences_per_source,seed):
    pad_id=int(getattr(tokenizer,'pad_token_id',0) or 0);eos_id=int(getattr(tokenizer,'eos_token_id',pad_id) or pad_id)
    rows=[];manifest=[]
    for source in sorted({document.source for document in documents}):
        collected=0
        for document in _stable_document_order([value for value in documents if value.source==source],seed):
            ids=tokenizer_encode(tokenizer,document_text(document))
            if len(ids)<2: continue
            maximum=max(0,len(ids)-(sequence_length+1));start=0 if maximum==0 else int(hashlib.sha256(f'{seed}:{document.uid}'.encode()).hexdigest()[:16],16)%(maximum+1)
            sequence=list(ids[start:start+sequence_length+1]);valid=min(sequence_length,len(sequence)-1)
            if len(sequence)<2: sequence.append(eos_id);valid=1
            sequence.extend([pad_id]*(sequence_length+1-len(sequence)));sequence=sequence[:sequence_length+1]
            labels=torch.tensor(sequence[1:],dtype=torch.long)
            if valid<sequence_length: labels[valid:]=-100
            rows.append({'input_ids':torch.tensor(sequence[:-1],dtype=torch.long),'labels':labels})
            manifest.append({'uid':document.uid,'source':source,'start_token':start,'valid_targets':valid});collected+=1
            if collected>=sequences_per_source: break
        if collected<sequences_per_source: raise ValueError(f'Only {collected} validation sequences available for {source}')
    return rows,manifest

def save_or_verify_json(path:Path,value:Any)->None:
    path=Path(path)
    if path.exists() and json.loads(path.read_text(encoding='utf-8'))!=value: raise RuntimeError(f'Persisted protocol changed: {path}')
    if not path.exists(): atomic_write_json(path,value)

def learning_rate_multiplier(step,total_steps,warmup_steps,min_ratio):
    if step<warmup_steps: return max(1,step+1)/max(1,warmup_steps)
    progress=min(1.0,(step-warmup_steps)/max(1,total_steps-warmup_steps));return min_ratio+(1-min_ratio)*0.5*(1+math.cos(math.pi*progress))
def choose_precision(device):
    if not str(device).startswith('cuda'): return None,'fp32',False
    if torch.cuda.is_bf16_supported(): return torch.bfloat16,'bf16',False
    return torch.float16,'fp16',True
def make_scaler(enabled):
    try: return torch.amp.GradScaler('cuda',enabled=enabled)
    except (AttributeError,TypeError): return torch.cuda.amp.GradScaler(enabled=enabled)
def scaler_update_succeeded(scale_before,scale_after):
    return float(scale_after)>=float(scale_before)
assert scaler_update_succeeded(1024,1024) and scaler_update_succeeded(1024,2048)
assert not scaler_update_succeeded(1024,512)
def atomic_copy(source:Path,destination:Path)->None:
    source,destination=Path(source),Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=destination.parent,suffix=destination.suffix,delete=False) as handle: temporary=Path(handle.name)
    try:
        shutil.copyfile(source,temporary)
        if sha256_file(source)!=sha256_file(temporary): raise IOError('Atomic copy verification failed')
        os.replace(temporary,destination)
    finally: temporary.unlink(missing_ok=True)

def save_inference_checkpoint(path,model,model_config,tokenizer_meta,dataset_fingerprint_value,step,validation_loss):
    state={'format':'phase1_inference_v1','created_utc':utc_now(),'model_state':model.state_dict(),'model_config':dict(model_config),
           'tokenizer_fingerprint':tokenizer_meta['fingerprint'],'dataset_fingerprint':dataset_fingerprint_value,
           'step':int(step),'validation_loss':None if validation_loss is None else float(validation_loss)}
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as handle: temporary=Path(handle.name)
    try:
        torch.save(state,temporary);check=torch.load(temporary,map_location='cpu',weights_only=False)
        if check.get('format')!='phase1_inference_v1' or check.get('step')!=int(step): raise RuntimeError('Inference checkpoint readback failed')
        os.replace(temporary,path)
    finally: temporary.unlink(missing_ok=True)
    return {'path':str(path),'sha256':sha256_file(path),'step':step}

def export_training_recovery_snapshot(root:Path)->Path:
    root=Path(root);output=DIRS['recovery']/'training_recovery.latest.zip';output.parent.mkdir(parents=True,exist_ok=True)
    include=[]
    for relative in ('model','tokenizer','manifests','histories'):
        include.extend(path for path in (root/relative).rglob('*') if path.is_file())
    for name in ('checkpoint.latest.pt','previous_known_good.pt','selected_model.pt','model_final.pt'):
        path=DIRS['checkpoints']/name
        if path.exists(): include.append(path)
    with tempfile.NamedTemporaryFile(dir=output.parent,suffix='.zip',delete=False) as handle: temporary=Path(handle.name)
    try:
        with zipfile.ZipFile(temporary,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
            for path in sorted(set(include)): archive.write(path,path.relative_to(root).as_posix())
        validate_zip(temporary);os.replace(temporary,output)
    finally: temporary.unlink(missing_ok=True)
    atomic_write_json(DIRS['recovery']/'training_recovery.latest.json',{'created_utc':utc_now(),'path':str(output),'sha256':sha256_file(output),'bytes':output.stat().st_size})
    return output

@torch.no_grad()
def evaluate_next_token_loss(model,loader,device,amp_dtype):
    was_training=model.training;model.eval();loss_sum=0.0;token_count=0
    for batch in loader:
        input_ids=batch['input_ids'].to(device);labels=batch['labels'].to(device)
        with torch.autocast(device_type=torch.device(device).type,dtype=amp_dtype,enabled=amp_dtype is not None):
            logits=_logits(model(input_ids));loss=torch.nn.functional.cross_entropy(logits.reshape(-1,logits.shape[-1]),labels.reshape(-1),ignore_index=-100,reduction='sum')
        loss_sum+=float(loss.cpu());token_count+=int((labels!=-100).sum())
    model.train(was_training)
    if not token_count: raise ValueError('Validation loader contained no targets')
    return loss_sum/token_count

def append_event(path:Path,event:Mapping[str,Any])->None:
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('ab') as handle: handle.write((json.dumps(dict(event))+'\n').encode());handle.flush();os.fsync(handle.fileno())

def train_phase1_model(model,tokenizer,train_docs,val_docs,model_config,tokenizer_meta,*,device='cuda:0',resume=True,training_config=None):
    cfg=dict(TRAINING_CONFIG if training_config is None else training_config)
    if cfg['num_workers']!=0: raise ValueError('num_workers must remain 0 for exact sampler commits')
    if not train_docs or not val_docs: raise ValueError('Nonempty train/validation data required')
    ratios=validate_source_mixture(cfg['source_mixture'],(document.source for document in train_docs))
    split_meta=json.loads((DIRS['manifests']/'document_split.json').read_text(encoding='utf-8'));dataset_fp=split_meta['dataset_fingerprint']
    dataset=MixedDocumentCausalDataset(train_docs,tokenizer,ratios,cfg['samples_per_epoch'],cfg['sequence_length'],cfg['seed'])
    sampler=StatefulRandomSampler(dataset,seed=cfg['seed']);loader=DataLoader(dataset,batch_size=cfg['batch_size'],sampler=sampler,num_workers=0,pin_memory=str(device).startswith('cuda'),drop_last=True)
    validation_rows,validation_manifest=build_fixed_validation_rows(val_docs,tokenizer,cfg['sequence_length'],cfg['validation_sequences_per_source'],cfg['seed'])
    save_or_verify_json(DIRS['manifests']/'validation_sequence_manifest.json',{'protocol':'equal sequences per source','rows':validation_manifest})
    validation_loader=DataLoader(validation_rows,batch_size=cfg['batch_size'],shuffle=False,num_workers=0)
    model=model.to(device)
    if hasattr(model,'set_gradient_checkpointing'): model.set_gradient_checkpointing(cfg['gradient_checkpointing'])
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg['learning_rate'],betas=tuple(cfg['betas']),eps=cfg['epsilon'],weight_decay=cfg['weight_decay'])
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:learning_rate_multiplier(step,cfg['max_steps'],cfg['warmup_steps'],cfg['min_lr_ratio']))
    amp_dtype,actual_precision,use_scaler=choose_precision(device);scaler=make_scaler(use_scaler);checkpoint_scaler=scaler if use_scaler else None
    history_path=DIRS['histories']/'training.jsonl';history=JsonlHistory(history_path);latest_path=DIRS['checkpoints']/'checkpoint.latest.pt'
    progress=ProgressState(cfg['experiment_id'],device,cfg['max_steps']);validation_history=[]
    if resume and latest_path.exists():
        preview=torch.load(latest_path,map_location='cpu',weights_only=False)
        if preview.get('training_config')!=cfg: raise RuntimeError('Checkpoint training config differs')
        del preview
        if cfg['auto_reconcile_strict_history_extension']:
            reconciliation=reconcile_history_to_checkpoint(latest_path,history_path,apply=True)
            if reconciliation['status']=='rolled_back': print('Reconciled uncommitted history tail:',reconciliation)
        state=restore_checkpoint(latest_path,model=model,optimizer=optimizer,scheduler=scheduler,scaler=checkpoint_scaler,sampler=sampler,
            expected_tokenizer_fingerprint=tokenizer_meta['fingerprint'],expected_dataset_fingerprint=dataset_fp,
            expected_experiment_id=cfg['experiment_id'],expected_model_config=model_config,history_path=history_path,device=device)
        progress=ProgressState(**state['progress']);validation_history=list(state.get('validation_history',[]));dataset.set_epoch(sampler.epoch)
    elif history.records(): raise RuntimeError('History exists without a matching resumed checkpoint')
    reporter=ProgressReporter(progress);iterator=iter(loader);best_loss=min((float(row['loss']) for row in validation_history),default=math.inf)
    try:
        while reporter.state.step<cfg['max_steps']:
            model.train();optimizer.zero_grad(set_to_none=True);microbatches=[];step_tokens=0
            for _ in range(cfg['gradient_accumulation_steps']):
                try: batch=next(iterator)
                except StopIteration:
                    sampler.next_epoch();dataset.set_epoch(sampler.epoch);iterator=iter(loader);batch=next(iterator)
                valid=int((batch['labels']!=-100).sum());step_tokens+=valid;microbatches.append(batch)
            if step_tokens<=0: raise ValueError('Accumulation window has no valid targets')
            loss_numerator=0.0
            for batch in microbatches:
                input_ids=batch['input_ids'].to(device);labels=batch['labels'].to(device)
                with torch.autocast(device_type=torch.device(device).type,dtype=amp_dtype,enabled=amp_dtype is not None):
                    logits=_logits(model(input_ids));loss_sum=torch.nn.functional.cross_entropy(logits.reshape(-1,logits.shape[-1]),labels.reshape(-1),ignore_index=-100,reduction='sum')
                    backward_loss=loss_sum/step_tokens
                if not torch.isfinite(loss_sum): raise FloatingPointError('Non-finite training loss')
                (scaler.scale(backward_loss) if use_scaler else backward_loss).backward();loss_numerator+=float(loss_sum.detach().cpu())
            if use_scaler: scaler.unscale_(optimizer)
            gradient_norm=float(torch.nn.utils.clip_grad_norm_(model.parameters(),cfg['gradient_clip_norm']).detach().cpu());learning_rate=float(optimizer.param_groups[0]['lr'])
            update_succeeded=True
            if use_scaler:
                previous_scale=float(scaler.get_scale());scaler.step(optimizer);scaler.update();update_succeeded=scaler_update_succeeded(previous_scale,scaler.get_scale())
            else: optimizer.step()
            if not update_succeeded:
                append_event(DIRS['histories']/'overflow_events.jsonl',{'attempted_step':reporter.state.step+1,'epoch':sampler.epoch,'scale_before':previous_scale,'scale_after':float(scaler.get_scale()),'created_utc':utc_now()})
                continue
            scheduler.step();step=reporter.state.step+1;tokens_processed=reporter.state.tokens_processed+step_tokens;mean_loss=loss_numerator/step_tokens
            validation_loss=None;improved=False
            if step%cfg['validation_interval']==0 or step==cfg['max_steps']:
                validation_loss=evaluate_next_token_loss(model,validation_loader,device,amp_dtype)
                validation_history.append({'step':step,'loss':validation_loss,'perplexity':float(math.exp(validation_loss)) if validation_loss<709 else math.inf})
                improved=validation_loss<best_loss;best_loss=min(best_loss,validation_loss)
            reporter.update(step=step,epoch=sampler.epoch,loss=mean_loss,learning_rate=learning_rate,tokens_processed=tokens_processed,gradient_norm=gradient_norm)
            history.append({'step':step,'epoch':sampler.epoch,'loss':mean_loss,'validation_loss':validation_loss,'learning_rate':learning_rate,
                'gradient_norm':gradient_norm,'tokens_processed':tokens_processed,'tokens_per_second':tokens_processed/max(reporter.state.cumulative_seconds,1e-9),
                'gpu_memory_gib':torch.cuda.max_memory_allocated(torch.device(device))/2**30 if str(device).startswith('cuda') else None,
                'precision':actual_precision,'created_utc':utc_now()})
            if step%cfg['checkpoint_interval']==0 or improved or step==cfg['max_steps']:
                reporter.state.last_checkpoint_step=step
                save_checkpoint_atomic(latest_path,model=model,optimizer=optimizer,scheduler=scheduler,scaler=checkpoint_scaler,sampler=sampler,
                    progress=asdict(reporter.state),model_config=model_config,tokenizer_fingerprint=tokenizer_meta['fingerprint'],
                    dataset_fingerprint_value=dataset_fp,training_config=cfg,history_path=history_path,validation_history=validation_history,
                    experiment_id=cfg['experiment_id'],device=device)
                if improved: save_inference_checkpoint(DIRS['checkpoints']/'selected_model.pt',model,model_config,tokenizer_meta,dataset_fp,step,validation_loss)
                if step%cfg['recovery_interval']==0: export_training_recovery_snapshot(ROOT)
        reporter.state.last_checkpoint_step=reporter.state.step
        save_checkpoint_atomic(latest_path,model=model,optimizer=optimizer,scheduler=scheduler,scaler=checkpoint_scaler,sampler=sampler,
            progress=asdict(reporter.state),model_config=model_config,tokenizer_fingerprint=tokenizer_meta['fingerprint'],dataset_fingerprint_value=dataset_fp,
            training_config=cfg,history_path=history_path,validation_history=validation_history,experiment_id=cfg['experiment_id'],device=device)
        final_validation=validation_history[-1]['loss'] if validation_history else None
        save_inference_checkpoint(DIRS['checkpoints']/'model_final.pt',model,model_config,tokenizer_meta,dataset_fp,reporter.state.step,final_validation)
        if not (DIRS['checkpoints']/'selected_model.pt').exists(): save_inference_checkpoint(DIRS['checkpoints']/'selected_model.pt',model,model_config,tokenizer_meta,dataset_fp,reporter.state.step,final_validation)
        if not (DIRS['checkpoints']/'previous_known_good.pt').exists(): atomic_copy(latest_path,DIRS['checkpoints']/'previous_known_good.pt')
        export_training_recovery_snapshot(ROOT)
    finally: reporter.close()
    result={'completed_step':reporter.state.step,'tokens_processed':reporter.state.tokens_processed,'best_validation_loss':best_loss,
            'actual_precision':actual_precision,'latest_checkpoint':str(latest_path),'selected_checkpoint':str(DIRS['checkpoints']/'selected_model.pt')}
    atomic_write_json(DIRS['manifests']/'training_result.json',result);return result

def run_gpu_training_smoke_test(device='cuda:0'):
    if not torch.cuda.is_available(): raise RuntimeError('GPU smoke test requires CUDA')
    cfg=dict(TRAINING_CONFIG);ratios=validate_source_mixture(cfg['source_mixture'],(document.source for document in TRAIN_DOCS))
    _,validation_manifest=build_fixed_validation_rows(VAL_DOCS,tokenizer,cfg['sequence_length'],cfg['validation_sequences_per_source'],cfg['seed'])
    save_or_verify_json(DIRS['manifests']/'validation_sequence_manifest.json',{'protocol':'equal sequences per source','rows':validation_manifest})
    samples=max(cfg['batch_size']*2,4);dataset=MixedDocumentCausalDataset(TRAIN_DOCS,tokenizer,ratios,samples,cfg['sequence_length'],cfg['seed'])
    sampler=StatefulRandomSampler(dataset,seed=cfg['seed']);loader=DataLoader(dataset,batch_size=cfg['batch_size'],sampler=sampler,num_workers=0,drop_last=True)
    smoke_model=RECONSTRUCT_MODEL(MODEL_CONFIG).to(device);smoke_model.set_gradient_checkpointing(cfg['gradient_checkpointing'])
    optimizer=torch.optim.AdamW(smoke_model.parameters(),lr=cfg['learning_rate'],betas=tuple(cfg['betas']),eps=cfg['epsilon'],weight_decay=cfg['weight_decay'])
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:learning_rate_multiplier(step,10,1,cfg['min_lr_ratio']))
    amp_dtype,precision,use_scaler=choose_precision(device);scaler=make_scaler(use_scaler);checkpoint_scaler=scaler if use_scaler else None
    batch=next(iter(loader));input_ids=batch['input_ids'].to(device);labels=batch['labels'].to(device);optimizer.zero_grad(set_to_none=True)
    torch.cuda.reset_peak_memory_stats(torch.device(device))
    with torch.autocast(device_type='cuda',dtype=amp_dtype,enabled=amp_dtype is not None):
        logits=_logits(smoke_model(input_ids));loss=torch.nn.functional.cross_entropy(logits.reshape(-1,logits.shape[-1]),labels.reshape(-1),ignore_index=-100)
    (scaler.scale(loss) if use_scaler else loss).backward()
    if use_scaler: scaler.unscale_(optimizer)
    grad_norm=float(torch.nn.utils.clip_grad_norm_(smoke_model.parameters(),cfg['gradient_clip_norm']).cpu())
    if use_scaler: before=float(scaler.get_scale());scaler.step(optimizer);scaler.update();updated=scaler_update_succeeded(before,scaler.get_scale())
    else: optimizer.step();updated=True
    if not updated: raise RuntimeError('Representative FP16 smoke update overflowed; lower memory/load or inspect numerics')
    scheduler.step();peak=torch.cuda.max_memory_allocated(torch.device(device))/2**30;loss_value=float(loss.detach().cpu())
    split_meta=json.loads((DIRS['manifests']/'document_split.json').read_text())
    with tempfile.TemporaryDirectory() as temporary_name:
        temporary=Path(temporary_name);history=JsonlHistory(temporary/'history.jsonl');history.append({'step':1,'loss':loss_value})
        checkpoint=temporary/'smoke.pt';progress=ProgressState('gpu_smoke',device,1,step=1,tokens_processed=int((labels!=-100).sum()),last_checkpoint_step=1)
        save_checkpoint_atomic(checkpoint,model=smoke_model,optimizer=optimizer,scheduler=scheduler,scaler=checkpoint_scaler,sampler=sampler,
            progress=asdict(progress),model_config=MODEL_CONFIG,tokenizer_fingerprint=TOKENIZER_META['fingerprint'],
            dataset_fingerprint_value=split_meta['dataset_fingerprint'],training_config={'smoke':True},history_path=history.path,
            validation_history=[],experiment_id='gpu_smoke',device=device)
        del logits,input_ids,labels,loss,optimizer,smoke_model;torch.cuda.empty_cache()
        clone=RECONSTRUCT_MODEL(MODEL_CONFIG);clone_optimizer=torch.optim.AdamW(clone.parameters(),lr=cfg['learning_rate'],betas=tuple(cfg['betas']),eps=cfg['epsilon'],weight_decay=cfg['weight_decay'])
        clone_scheduler=torch.optim.lr_scheduler.LambdaLR(clone_optimizer,lambda step:learning_rate_multiplier(step,10,1,cfg['min_lr_ratio']))
        clone_scaler=make_scaler(use_scaler);clone_sampler=StatefulRandomSampler(dataset,seed=cfg['seed'])
        restored=restore_checkpoint(checkpoint,model=clone,optimizer=clone_optimizer,scheduler=clone_scheduler,scaler=clone_scaler if use_scaler else None,
            sampler=clone_sampler,expected_tokenizer_fingerprint=TOKENIZER_META['fingerprint'],expected_dataset_fingerprint=split_meta['dataset_fingerprint'],
            expected_experiment_id='gpu_smoke',expected_model_config=MODEL_CONFIG,history_path=history.path,device=None)
        if restored['progress']['step']!=1 or not clone_optimizer.state: raise AssertionError('Checkpoint did not restore model/optimizer progress')
    report={'status':'PASS','loss':loss_value,'gradient_norm':grad_norm,'precision':precision,'peak_gpu_memory_gib':peak,
            'sequence_length':cfg['sequence_length'],'batch_size':cfg['batch_size'],'optimizer_restore':'PASS','created_utc':utc_now()}
    atomic_write_json(DIRS['manifests']/'gpu_training_smoke_test.json',report);return report

STUDENT_TRAIN_FUNCTION=train_phase1_model
RUN_GPU_TRAINING_SMOKE_TEST=False
GPU_TRAINING_SMOKE_RESULT=run_gpu_training_smoke_test('cuda:0') if RUN_GPU_TRAINING_SMOKE_TEST else None

def preflight_full_training():
    problems=[]
    if not CFG.full_training_authorized: problems.append('CFG.full_training_authorized is False')
    if tokenizer is None or TOKENIZER_META is None: problems.append('tokenizer unavailable')
    if not TRAIN_DOCS or not VAL_DOCS: problems.append('official data unavailable')
    required=[DIRS['manifests']/name for name in ('document_split.json','model_validation.json','gpu_training_smoke_test.json','validation_sequence_manifest.json')]
    problems.extend(f'missing {path}' for path in required if not path.exists())
    if problems: raise RuntimeError('Full-training preflight failed:\n- '+'\n- '.join(problems))

RUN_FULL_TRAINING=False
if RUN_FULL_TRAINING:
    preflight_full_training();TRAINING_RESULT=STUDENT_TRAIN_FUNCTION(MODEL,tokenizer,TRAIN_DOCS,VAL_DOCS,MODEL_CONFIG,TOKENIZER_META,device='cuda:0',resume=True,training_config=TRAINING_CONFIG)
else: TRAINING_RESULT=None;print('Run the GPU smoke test first; then enable both explicit full-training gates.')


## 11. Local evaluation plumbing

In [ ]:
EXPECTED_METRIC_FIELDS=('bertscore_f1','rouge_l_f1','bleu_4')
METRIC_ALIASES={
    'bertscore_f1':'bertscore_f1','bert_score_f1':'bertscore_f1','bertscore':'bertscore_f1',
    'rouge_l_f1':'rouge_l_f1','rougel_f1':'rouge_l_f1','rouge-l_f1':'rouge_l_f1',
    'bleu_4':'bleu_4','bleu4':'bleu_4','bleu-4':'bleu_4',
}
def perplexity_from_loss(loss): return float(math.exp(loss)) if loss<709 else math.inf
def normalize_official_metrics(raw:Mapping[str,Any])->dict[str,float]:
    normalized={}
    for key,value in raw.items():
        canonical=METRIC_ALIASES.get(str(key).strip().lower().replace(' ','_'))
        if canonical: normalized[canonical]=float(value)
    missing=[key for key in EXPECTED_METRIC_FIELDS if key not in normalized]
    if missing: raise ValueError(f'Official evaluator result is missing required metrics {missing}; raw keys={list(raw)}')
    return {key:normalized[key] for key in EXPECTED_METRIC_FIELDS}
def run_official_evaluator(evaluator,predictions,references,checkpoint_id):
    if len(predictions)!=len(references) or not predictions: raise ValueError('Predictions/references must be nonempty and aligned')
    metrics=normalize_official_metrics(evaluator(predictions,references));record={'checkpoint_id':checkpoint_id,'created_utc':utc_now(),'metrics':metrics,'examples':len(predictions)}
    atomic_write_json(DIRS['evaluations']/f'{checkpoint_id}.json',record);csv_path=DIRS['evaluations']/'metrics.csv';write_header=not csv_path.exists()
    expected_header=['checkpoint_id','created_utc','examples',*EXPECTED_METRIC_FIELDS]
    if csv_path.exists():
        with csv_path.open('r',newline='',encoding='utf-8') as existing_handle:
            existing_header=next(csv.reader(existing_handle),None)
        if existing_header!=expected_header: raise RuntimeError(f'Existing metrics.csv schema differs: {existing_header}')
    with csv_path.open('a',newline='',encoding='utf-8') as handle:
        writer=csv.DictWriter(handle,fieldnames=expected_header)
        if write_header: writer.writeheader()
        writer.writerow({'checkpoint_id':checkpoint_id,'created_utc':record['created_utc'],'examples':len(predictions),**metrics})
    return record

def load_official_evaluator(path:Path,function_name:str,argument_order:str):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(path)
    if not function_name: raise ValueError('Set CS781_EVALUATOR_FUNCTION to the inspected official callable name')
    if argument_order not in ('predictions_references','references_predictions'): raise ValueError('Set CS781_EVALUATOR_ARGUMENT_ORDER explicitly')
    module=import_source_module(f'cs781_official_evaluator_{sha256_file(path)[:12]}',path);function=getattr(module,function_name,None)
    if not callable(function): raise AttributeError(f'{path} has no callable {function_name!r}')
    def adapted(predictions,references):
        result=function(predictions,references) if argument_order=='predictions_references' else function(references,predictions)
        if not isinstance(result,Mapping): raise TypeError(f'Official evaluator must return a mapping, got {type(result).__name__}')
        return result
    return adapted

def discover_evaluator_candidates(root:Path)->list[Path]:
    if not Path(root).exists(): return []
    return sorted(path for path in Path(root).rglob('*.py') if any(term in path.name.lower() for term in ('eval','metric','score')) and ROOT.resolve() not in path.resolve().parents)

OFFICIAL_EVALUATOR_PATH=os.environ.get('CS781_EVALUATOR_PATH');OFFICIAL_EVALUATOR_FUNCTION=os.environ.get('CS781_EVALUATOR_FUNCTION')
OFFICIAL_EVALUATOR_ARGUMENT_ORDER=os.environ.get('CS781_EVALUATOR_ARGUMENT_ORDER')
OFFICIAL_EVALUATOR=(load_official_evaluator(Path(OFFICIAL_EVALUATOR_PATH),OFFICIAL_EVALUATOR_FUNCTION,OFFICIAL_EVALUATOR_ARGUMENT_ORDER) if OFFICIAL_EVALUATOR_PATH else None)
if OFFICIAL_EVALUATOR is None:
    EVALUATOR_CANDIDATES=discover_evaluator_candidates(Path(CFG.input_root));print('Inspect official evaluator candidates:',[str(path) for path in EVALUATOR_CANDIDATES])
    print('Set CS781_EVALUATOR_PATH, CS781_EVALUATOR_FUNCTION and CS781_EVALUATOR_ARGUMENT_ORDER explicitly.')


In [ ]:
def plot_saved_history(history_path: Path, output_path: Path) -> None:
    import matplotlib.pyplot as plt
    records = JsonlHistory(history_path).records()
    if not records:
        raise ValueError("No history records to plot")
    figure, axes = plt.subplots(2, 2, figsize=(12, 8))
    fields = [("loss", "Training loss"), ("learning_rate", "Learning rate"), ("tokens_per_second", "Tokens/s"), ("gpu_memory_gib", "GPU memory GiB")]
    for axis, (field_name, title) in zip(axes.flat, fields):
        x, y = zip(*[(row["step"], row[field_name]) for row in records if row.get(field_name) is not None]) if any(row.get(field_name) is not None for row in records) else ([], [])
        axis.plot(x, y)
        axis.set_title(title)
        axis.set_xlabel("Optimizer step")
        axis.grid(alpha=0.25)
    figure.tight_layout()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, dpi=160)
    plt.close(figure)

# plot_saved_history(DIRS["histories"] / "training.jsonl", DIRS["plots"] / "training_history.png")

## 12. Generation and prediction management

In [ ]:
WORD_PATTERN=re.compile(r'\S+')
def words(text:str)->list[str]: return WORD_PATTERN.findall(text.strip())
def exact_word_prefix(text:str,count:int=128)->str:
    units=words(text)
    if len(units)<count: raise ValueError(f'Only {len(units)} words, need {count}')
    return ' '.join(units[:count])
def infer_context_limit(model,explicit_limit=None):
    if explicit_limit is not None: return int(explicit_limit)
    config=getattr(model,'config',None)
    for key in ('max_position_embeddings','n_positions','block_size','context_length'):
        result=getattr(config,key,None)
        if isinstance(result,int) and result>0: return result
    return int(CFG.architecture_record['context_length'])

@torch.inference_mode()
def greedy_continuation(model,tokenizer,prompt,device,*,target_words=128,max_total_new_tokens=2048,context_limit=None):
    model=model.to(device);model.eval();limit=infer_context_limit(model,context_limit);prompt_ids=tokenizer_encode(tokenizer,prompt)
    if not prompt_ids: raise ValueError('Prompt encoded to no tokens')
    context=list(prompt_ids[-limit:]);generated=[];eos=getattr(tokenizer,'eos_token_id',None)
    output=model(torch.tensor([context],dtype=torch.long,device=device),use_cache=True);logits,past=output
    for _ in range(max_total_new_tokens):
        next_id=int(logits[0,-1].argmax());generated.append(next_id);context.append(next_id)
        continuation=tokenizer_decode(tokenizer,generated)
        if len(words(continuation))>=target_words+1: return exact_word_prefix(continuation,target_words)
        if eos is not None and next_id==eos:
            if len(words(continuation))==target_words: return exact_word_prefix(continuation,target_words)
            break
        if len(context)<=limit:
            logits,past=model(torch.tensor([[next_id]],dtype=torch.long,device=device),past_key_values=past,use_cache=True)
        else:
            context=context[-limit:];logits,past=model(torch.tensor([context],dtype=torch.long,device=device),use_cache=True)
    raise RuntimeError(f'Generation ended before {target_words} verified words (generated {len(words(tokenizer_decode(tokenizer,generated)))}).')

def load_test_records(path:Path,id_field:str,prompt_field:str)->list[dict[str,str]]:
    if not id_field or not prompt_field: raise ValueError('Verified test fields are required')
    result=[];seen=set()
    for index,row in enumerate(iter_json_records(path,id_field=id_field,text_field=prompt_field)):
        if id_field not in row or prompt_field not in row: raise KeyError(f'Test row {index} missing {id_field}/{prompt_field}')
        identifier=str(row[id_field]);prompt=normalize_text(row[prompt_field])
        if identifier in seen: raise ValueError(f'Duplicate official ID: {identifier}')
        seen.add(identifier);result.append({'id':identifier,'prompt':prompt})
    return result
def test_records_fingerprint(records):
    digest=hashlib.sha256()
    for row in records: digest.update(f"{row['id']}\t{hashlib.sha256(row['prompt'].encode()).hexdigest()}\n".encode())
    return digest.hexdigest()


In [ ]:
def generate_official_predictions(model,tokenizer,test_records,device,*,checkpoint_id,tokenizer_meta):
    safe=re.sub(r'[^A-Za-z0-9_.-]+','_',checkpoint_id);partial_path=DIRS['predictions']/f'official_predictions.{safe}.partial.json';meta_path=partial_path.with_suffix('.meta.json')
    expected_meta={'checkpoint_id':checkpoint_id,'tokenizer_fingerprint':tokenizer_meta['fingerprint'],'test_fingerprint':test_records_fingerprint(test_records),'decoding':dict(CFG.decoding)}
    predictions={};failures=[]
    if partial_path.exists():
        if not meta_path.exists() or json.loads(meta_path.read_text(encoding='utf-8'))!=expected_meta: raise RuntimeError('Partial predictions do not authenticate the current checkpoint/tokenizer/test/decoding configuration')
        predictions=json.loads(partial_path.read_text(encoding='utf-8'));expected={row['id'] for row in test_records}
        if set(predictions)-expected: raise RuntimeError('Partial prediction file contains foreign IDs')
        for key,value in predictions.items():
            if not isinstance(value,dict) or set(value)!={'text'} or len(words(value['text']))!=CFG.decoding['target_words']: raise RuntimeError(f'Invalid cached prediction: {key}')
    else: atomic_write_json(meta_path,expected_meta)
    for row in tqdm(test_records,desc='Official test generation'):
        if row['id'] in predictions: continue
        try:
            continuation=greedy_continuation(model,tokenizer,row['prompt'],device,target_words=CFG.decoding['target_words'],max_total_new_tokens=CFG.decoding['max_total_new_tokens'])
            predictions[row['id']]={'text':continuation};atomic_write_json(partial_path,predictions)
        except Exception as exc: failures.append({'id':row['id'],'error':f'{type(exc).__name__}: {exc}'})
    atomic_write_json(DIRS['predictions']/'generation_failures.json',failures)
    if failures: raise RuntimeError(f'Generation failed for {len(failures)} examples')
    provenance={**expected_meta,'prediction_count':len(predictions),'partial_prediction_sha256':sha256_file(partial_path),'created_utc':utc_now()}
    atomic_write_json(DIRS['predictions']/'inference_provenance.json',provenance);return predictions,failures

def load_selected_model_for_inference(checkpoint_path,tokenizer_meta,reconstruct_model,device):
    state=torch.load(Path(checkpoint_path),map_location='cpu',weights_only=False)
    if state.get('tokenizer_fingerprint')!=tokenizer_meta['fingerprint']: raise RuntimeError('Selected checkpoint tokenizer mismatch')
    restored=reconstruct_model(state['model_config']);restored.load_state_dict(state['model_state'],strict=True);return restored.to(device).eval(),state
def evaluate_local_continuations(model,tokenizer,examples,evaluator,device,checkpoint_id):
    if evaluator is None: raise RuntimeError('Load the inspected professor-supplied evaluator first')
    predictions=[];references=[]
    for row in tqdm(examples,desc='Fixed development generation'):
        predictions.append(greedy_continuation(model,tokenizer,row['prompt'],device,target_words=CFG.decoding['target_words'],max_total_new_tokens=CFG.decoding['max_total_new_tokens']));references.append(row['reference'])
    return run_official_evaluator(evaluator,predictions,references,checkpoint_id)
def run_phase1_inference_pipeline(device='cuda:0'):
    checkpoint_path=DIRS['checkpoints']/'selected_model.pt'
    if not checkpoint_path.exists(): raise FileNotFoundError('Train/select a checkpoint before inference')
    if not LOCAL_DEV: raise RuntimeError('Fixed LOCAL_DEV examples unavailable')
    selected_model,state=load_selected_model_for_inference(checkpoint_path,TOKENIZER_META,RECONSTRUCT_MODEL,device);checkpoint_id=f"selected-{sha256_file(checkpoint_path)[:12]}"
    evaluation=evaluate_local_continuations(selected_model,tokenizer,LOCAL_DEV,OFFICIAL_EVALUATOR,device,checkpoint_id)
    test_records=load_test_records(require_unique_file(DISCOVERED,CFG.test_file),CFG.test_id_field,CFG.test_prompt_field)
    predictions,failures=generate_official_predictions(selected_model,tokenizer,test_records,device,checkpoint_id=checkpoint_id,tokenizer_meta=TOKENIZER_META)
    return selected_model,evaluation,test_records,predictions,failures
RUN_INFERENCE=False
if RUN_INFERENCE: MODEL,EVAL_RECORD,TEST_RECORDS,PREDICTIONS,FAILURES=run_phase1_inference_pipeline('cuda:0')
else: print('Evaluation/generation pipeline is connected and waiting for RUN_INFERENCE=True.')


## 13. Codabench validation and packaging

In [ ]:
def validate_predictions(predictions: Any, official_ids: Sequence[str], target_words: int = 128) -> dict[str, Any]:
    errors = []
    if not isinstance(predictions, dict):
        return {"passed": False, "errors": ["prediction root must be an object"]}
    expected, actual = set(map(str, official_ids)), set(map(str, predictions))
    if missing := sorted(expected - actual):
        errors.append(f"missing IDs: {missing[:20]}")
    if unknown := sorted(actual - expected):
        errors.append(f"unknown IDs: {unknown[:20]}")
    for identifier in sorted(expected & actual):
        entry = predictions[identifier]
        if not isinstance(entry, dict) or set(entry) != {"text"}:
            errors.append(f"{identifier}: entry must contain only a text field")
            continue
        text = entry["text"]
        if not isinstance(text, str) or not text.strip():
            errors.append(f"{identifier}: empty/non-string text")
        elif len(words(text)) != target_words:
            errors.append(f"{identifier}: {len(words(text))} words, expected {target_words}")
    return {"passed": not errors, "errors": errors, "expected_ids": len(expected), "actual_ids": len(actual)}

def create_submission_zip(predictions: Mapping[str, Any], official_ids: Sequence[str], output_dir: Path) -> tuple[Path, dict[str, Any]]:
    report = validate_predictions(predictions, official_ids, CFG.decoding["target_words"])
    if not report["passed"]:
        raise ValueError("Submission validation failed:\n" + "\n".join(report["errors"]))
    provenance_path=DIRS['predictions']/'inference_provenance.json'
    if not provenance_path.exists(): raise RuntimeError('Missing inference provenance')
    provenance=json.loads(provenance_path.read_text(encoding='utf-8'))
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = output_dir / "prediction.json"
    atomic_write_json(prediction_path, predictions)
    zip_path = output_dir / "submission.zip"
    with tempfile.NamedTemporaryFile(dir=output_dir, suffix=".zip", delete=False) as handle:
        temporary = Path(handle.name)
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as archive:
            archive.write(prediction_path, "prediction.json")
        zip_report = validate_zip(temporary)
        if zip_report["members"] != ["prediction.json"]:
            raise ValueError(f"Invalid ZIP root structure: {zip_report['members']}")
        os.replace(temporary, zip_path)
    finally:
        temporary.unlink(missing_ok=True)
    report['prediction_sha256']=sha256_file(prediction_path)
    report['inference_provenance']=provenance
    report["zip"] = validate_zip(zip_path)
    atomic_write_json(output_dir / "submission_validation.json", report)
    return zip_path, report

if 'PREDICTIONS' in globals() and 'TEST_RECORDS' in globals():
    SUBMISSION_ZIP,SUBMISSION_REPORT=create_submission_zip(PREDICTIONS,[row['id'] for row in TEST_RECORDS],DIRS['submission']);print('Created validated Codabench package:',SUBMISSION_ZIP)
else: SUBMISSION_ZIP=SUBMISSION_REPORT=None;print('Submission packaging deferred until authenticated official predictions exist.')


## 14. Recovery archive and integrity manifest

In [ ]:
def build_integrity_manifest(root:Path,excluded:set[Path]|None=None)->dict[str,Any]:
    root=Path(root).resolve();excluded={Path(p).resolve() for p in (excluded or set())}
    files=[]
    for p in sorted(item for item in root.rglob('*') if item.is_file()):
        resolved=p.resolve()
        if resolved in excluded or any(item.is_dir() and (item==resolved or item in resolved.parents) for item in excluded) or p.is_symlink():continue
        files.append({'path':p.relative_to(root).as_posix(),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
    return {'created_utc':utc_now(),'files':files}
def create_recovery_zip(root:Path)->tuple[Path,dict[str,Any]]:
    root=Path(root).resolve();outdir=root/'recovery';outdir.mkdir(parents=True,exist_ok=True)
    out=outdir/'phase1_submission01_recovery.zip';previous=outdir/'phase1_submission01_recovery.previous.zip'
    manifest_path=root/'manifests'/'recovery_integrity.json'
    # Exclude previous archives, validation reports, and manifest from its own hash list.
    excluded={out,previous,manifest_path,outdir/'recovery_validation.json',root/'data'/'document_store'}
    manifest=build_integrity_manifest(root,excluded)
    atomic_write_json(manifest_path,manifest)
    with tempfile.NamedTemporaryFile(dir=outdir,suffix='.zip',delete=False) as h: tmp=Path(h.name)
    excluded.add(tmp)
    try:
        with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as z:
            for item in manifest['files']: z.write(root/item['path'],item['path'])
            z.write(manifest_path,'manifests/recovery_integrity.json')
        report=verify_recovery_archive(tmp)
        if out.exists():
            verify_recovery_archive(out)
            with tempfile.NamedTemporaryFile(dir=outdir,suffix='.zip',delete=False) as h: prev_tmp=Path(h.name)
            try: shutil.copyfile(out,prev_tmp);os.replace(prev_tmp,previous)
            finally: prev_tmp.unlink(missing_ok=True)
        os.replace(tmp,out);report=verify_recovery_archive(out)
        atomic_write_json(outdir/'recovery_validation.json',report)
        return out,report
    finally: tmp.unlink(missing_ok=True)
# RECOVERY_ZIP,RECOVERY_REPORT=create_recovery_zip(ROOT)

## 15. Final PASS/FAIL verification

In [ ]:
def final_verification(cfg,model=None,tokenizer_meta=None,official_ids=None):
    checks={};checks['trained_from_scratch_attested']=bool(cfg.trained_from_scratch_attestation)
    path=DIRS['manifests']/'model_validation.json';report=json.loads(path.read_text()) if path.exists() else {}
    checks['parameter_count_50m_to_300m']=50_000_000<=int(report.get('total',-1))<=300_000_000
    checks['gpu_training_smoke_passed']=json.loads((DIRS['manifests']/'gpu_training_smoke_test.json').read_text()).get('status')=='PASS' if (DIRS['manifests']/'gpu_training_smoke_test.json').exists() else False
    meta_path=DIRS['tokenizer']/'tokenizer_metadata.json';checks['tokenizer_registered']=meta_path.exists()
    checks['tokenizer_integrity']=bool(meta_path.exists() and json.loads(meta_path.read_text()).get('fingerprint')==directory_fingerprint(DIRS['tokenizer']))
    checks['split_manifest_present']=(DIRS['manifests']/'document_split.json').exists();checks['validation_protocol_present']=(DIRS['manifests']/'validation_sequence_manifest.json').exists()
    checks['training_history_present']=(DIRS['histories']/'training.jsonl').exists();checks['evaluation_records_present']=any(DIRS['evaluations'].glob('*.json'))
    selected=DIRS['checkpoints']/'selected_model.pt';checks['selected_model_present']=selected.exists();checks['final_model_present']=(DIRS['checkpoints']/'model_final.pt').exists()
    checks['student_architecture_source_present']=any(DIRS['model'].glob('*.py'));submission=DIRS['submission']/'submission.zip';checks['submission_zip_valid']=False;checks['artifact_provenance_fresh']=False
    if submission.exists() and official_ids is not None:
        try:
            with zipfile.ZipFile(submission) as archive:
                if archive.namelist()==['prediction.json']:
                    predictions=json.loads(archive.read('prediction.json'));checks['submission_zip_valid']=validate_predictions(predictions,official_ids,cfg.decoding['target_words'])['passed']
            validation=json.loads((DIRS['submission']/'submission_validation.json').read_text());provenance=validation.get('inference_provenance',{})
            checks['artifact_provenance_fresh']=bool(selected.exists() and provenance.get('checkpoint_id')==f"selected-{sha256_file(selected)[:12]}" and validation.get('prediction_sha256')==sha256_file(DIRS['submission']/'prediction.json'))
        except Exception: pass
    recovery=DIRS['recovery']/'phase1_submission01_recovery.zip';checks['recovery_zip_valid']=False;checks['recovery_is_fresh']=False
    if recovery.exists():
        try:
            verify_recovery_archive(recovery);checks['recovery_zip_valid']=True
            inputs=[path for path in (selected,submission) if path.exists()];checks['recovery_is_fresh']=bool(inputs and recovery.stat().st_mtime>=max(path.stat().st_mtime for path in inputs))
        except Exception: pass
    checks['overall_inputs_available']=official_ids is not None and model is not None and tokenizer_meta is not None
    outcome={'overall':'PASS' if all(checks.values()) else 'FAIL','checks':checks,'created_utc':utc_now()};atomic_write_json(DIRS['manifests']/'final_verification.json',outcome);print(json.dumps(outcome,indent=2));return outcome
# Run only after rebuilding the final recovery archive:
# FINAL_REPORT=final_verification(CFG,MODEL,TOKENIZER_META,[row['id'] for row in TEST_RECORDS])


## Kaggle execution checklist

1. Enable Kaggle Internet. In Cell 9 set `INSTALL_RUNTIME_DEPENDENCIES=True` and `DOWNLOAD_PHASE1_FROM_HF=True` for the first download, or attach a private Kaggle dataset and leave download disabled. Never paste a Hugging Face token; use the `HF_TOKEN` Kaggle secret when the repository requires authentication.
2. Inspect the printed repository paths, resolved local paths, file sizes, layouts, and sampled schemas. Multiple copies require an explicit `CFG.official_file_overrides` entry.
3. Cell 14 preprocesses the large JSON streams into resumable compressed document stores, freezes a source-stratified split, records per-source document/word counts, and creates a fixed 16-example-per-source generation subset. Source ratios mean document draws, not token percentages.
4. Cell 16 restores a registered tokenizer before considering training. If no tokenizer exists, review the configuration and set `RUN_TOKENIZER_TRAINING=True` once.
5. Run model tests, then set `RUN_GPU_TRAINING_SMOKE_TEST=True` in Cell 29. Full training is blocked until a real 1024-token forward/backward/AdamW/checkpoint-restore smoke test passes on the selected Kaggle GPU.
6. After reviewing the smoke report and available RAM/GPU/disk, set `CFG.full_training_authorized=True` and `RUN_FULL_TRAINING=True`. FP16 overflow skips neither scheduler state nor completed-step accounting; accumulation is weighted by valid targets; document draws are resampled each epoch.
7. Download `recovery/training_recovery.latest.zip` at milestones. It is an automatic in-session recovery artifact, not persistent storage by itself.
8. Inspect the official evaluator and set all three evaluator environment variables explicitly. The CSV accepts only BERTScore F1, ROUGE-L F1 and BLEU-4 after validated name normalization.
9. Enable inference. Partial predictions are reusable only when checkpoint, tokenizer, test prompts and decoding configuration all match. Generation uses a verified KV cache until the 1024-token window must slide.
10. Package `submission.zip`, rebuild the final recovery archive, attest scratch training only when true, and require final verification `PASS` before uploading the ZIP to Codabench.
